# Reproducing Pelot et al. (2020), quantified morphology of the human, pig and rat vagus nerves

Paper. Pelot NA, Goldhagen GB, Cariello JE, Musselman ER, Clissold KA, Ezzell JA, Grill WM (2020).
"Quantified Morphology of the Cervical and Subdiaphragmatic Vagus Nerves of Human, Pig, and Rat".
*Front. Neurosci.* 14:601479, doi:10.3389/fnins.2020.601479.

Claims. `claims/pelot-2020.yaml`, written from the version of record (Europe PMC full text,
PMC7672126: main text, Methods, legends; no supplement, Eqs. 1 to 4 missing from the text).

Datasets and versions. Seven SPARC datasets, each read anonymously from the public bucket at the
object versions listed by the version's `manifest.json`:

| Folder | SPARC id | Version | DOI | What it holds |
|---|---|---|---|---|
| sparc-60 | 60 | 5 | 10.26275/lg3c-1kjf | rat MT traces (binary masks), code, output metrics |
| sparc-64 | 64 | 4 | 10.26275/maq2-eii4 | pig MT traces and micrographs, code |
| sparc-65 | 65 | 8 | 10.26275/qgja-q7zd | human anti-claudin-1 micrographs and traces, code, output metrics, Sunderland data |
| sparc-89 | 89 | 1 | 10.26275/8pc2-rhu2 | pig anti-fibronectin micrographs and traces, code |
| sparc-16 | 16 | 5 | 10.26275/z3ab-7j9y | rat MT micrographs |
| sparc-61 | 61 | 3 | 10.26275/sydt-lkiw | human MT micrographs |
| sparc-82 | 82 | 1 | 10.26275/pgr9-bk2e | pig MT micrographs |

Why these versions: the text of the paper that was read has no reference list, so it cites no
version of any dataset. Each is the most recent published version, the one the registry's target
gives. For sparc-60 the rat traces changed after version 1 (2020-02-28); versions 2 to 5 hold the
same masks. The paper's rat perineurium fit (22 fascicles, R2 = 0.77) is reproduced by the current
masks and not by those of version 1, while the output metrics file deposited in the datasets was
computed from version 1. The analysis uses version 5 and shows version 1 as a variant where the rat
traces matter. For sparc-64 the fascicle masks of version 1 also differ from version 4 (fascicle
counts); version 4 matches the paper's counts (data check below).

Method sources. The paper links no code, but the datasets hold the authors' MATLAB code
(`files/code/` of sparc-60, 64, 65 and 89). Where the Methods leave a detail open, it is taken from
that code, cited in `pelot.py`. The mixed models are refitted in Python (`mixed.py`), since JMP is
not available, with Kenward-Roger degrees of freedom, JMP's method (implemented in `mixed.py` and
checked on synthetic balanced designs, where it must give the classical split-plot tests). No
result depends on random draws; the synthetic check uses a fixed seed, a standard choice that
changes no result.

Claims tested and blocked.

| Claim | Status |
|---|---|
| rat-sample-set, human-sample-set, pig-mt-sample-set, pig-if-sample-set, mt-micrograph-collections | tested (link) |
| nerve-diameter-anova, human-pig-nerve-diameter-similar, rat-nerve-diameter, rat-ten-times-smaller, nerve-cervical-larger | tested |
| fascicle-number-human-pig, rat-subdiaphragmatic-multifascicular, rat-multifascicular-proportions | tested |
| fascicle-diameter-anova, rat-fascicle-diameter, cervical-fascicle-diameter-species, subdiaphragmatic-fascicle-diameter-comparable, fascicles-larger-cervical, rat-diameter-like-fascicle | tested |
| human-pig-tissue-proportions, pig-if-endoneurium-lower, rat-level-ratios, human-pig-level-ratios, human-variability, pig-oblong-human-round | tested |
| human-subdiaphragmatic-fatty-tissue, pig-cervical-cell-bodies, rat-carotid-sheath-bundles | tested by eye on the micrographs shown |
| literature-comparison-human-pig, rat-literature-comparison | tested on the literature values of Figure 7 as the paper's plotting code (DataPlotting_v4.m) holds them |
| perineurium-species-order, perineurium-ratio-decreases, human-perineurium-fit, pig-perineurium-fit, rat-perineurium-fit, perineurium-thicker-larger-fascicles, perineurium-no-level-difference | tested |
| sunderland-reanalysis | tested on the Sunderland and Bradley (1952) values deposited in sparc-65 |
| rat-perineurium-near-3-percent, human-pig-perineurium-thicker, pig-if-mt-consistent, peanut-fascicle-counts, no-age-trend | tested |

In [ ]:
import time, resource, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from scipy import stats
from scipy import ndimage as ndi
import matplotlib.pyplot as plt
import pelot, mixed

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
LEVEL = {"C": "cervical", "S": "subdiaphragmatic"}

## Loading

Every analysed sample is read from its binary masks (nerve, outer perineurium, inner perineurium, or
the rat perineurial rings) and measured as the paper's code does: boundaries as `bwboundaries`
traces them, areas by `polyarea`, effective diameters of the circle of equal area, perineurium
thickness half the difference of the outer and inner effective diameters. The sample lists are the
Excel files the paper's code reads. Rat samples are loaded a second time from version 1 of
sparc-60, for the variants.

In [ ]:
tab, res = pelot.load_all()
tab_v1, res_v1 = pelot.load_all(rat_version=1, only=["rat"])
print(f"loaded in {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
MAIN = tab[tab.species.isin(["human", "pig", "rat"])].copy()
IF = tab[tab.species == "pig_if"].copy()

## Data check

What was loaded, the units and ranges of every variable used, and the measurements checked against
the output metrics the authors deposited (`files/derivative/CadaverVNMorphology_OutputMetrics.mat`
in sparc-65, the same file as `RatVNMorphology_OutputMetrics.mat` in sparc-60, and
`files/code/PigIFMorphology.mat` in sparc-89). The data are static images, so there is no clock to
check.

In [ ]:
print("Samples by species, level and sex (analysed sets, from the code's sample lists and sparc-89):")
print(tab.groupby(["species", "level", "sex"]).size().unstack(fill_value=0))
print("\nSubjects per species:", tab.groupby("species").subject.nunique().to_dict())
print("\nScale (um per pixel) by species, from the scale bar masks:")
sc = pd.DataFrame([(k, v["um_per_px"]) for k, v in res.items()], columns=["key", "um_per_px"])
sc["species"] = [("pig_if" if k.startswith("IF") else {"C": "human", "P": "pig", "R": "rat"}[k[0]]) for k in sc.key]
print(sc.groupby("species").um_per_px.agg(["min", "max", "count"]).round(4))
print("\nRanges per sample (um, um^2):")
print(tab.groupby("species")[["nerve_d", "n_fasc", "endo_area"]].agg(["min", "max"]).round(1))
ph = pd.Series({k: v for k, v in pelot.MASK_PHOTOMETRIC.items()})
print("\nMask files read:", len(ph), "; stored with photometric MinIsWhite (inverted on reading):",
      int((ph == 0).sum()), "by dataset", ph[ph == 0].groupby(level=0).size().to_dict())
print("Nerve masks with more than one object:", [k for k, v in res.items() if v.get("n_nerve_objects", 1) > 1])
print("Inner traces not inside any outer trace:", int(sum((v["fasc"].get("outer", pd.Series(dtype=int)) < 0).sum() for v in res.values())))
print("Hole traces counted as inner traces (bwboundaries):", {k: v["holes_inner"] for k, v in res.items() if v.get("holes_inner")})

In [ ]:
# Check against the output metrics deposited with the data
om = pelot.read_mat("sparc-65", "files/derivative/CadaverVNMorphology_OutputMetrics.mat")
rows = []
for s, dn, nf, df_ in zip(om["sub_sam"], om["dNerve_um"], om["NFasc"], om["dFasc_um"]):
    df_ = np.sort(np.atleast_1d(df_).astype(float))
    for ver, t, r in [("v5", tab, res)] + ([("v1", tab_v1, res_v1)] if s[0] == "R" else []):
        row = t[t["sample"] == s].iloc[0]
        mine = np.sort(r[s]["fasc"].d_in.to_numpy())
        rows.append(dict(sample=s, version=ver, dnerve=row.nerve_d, dnerve_file=dn, nfasc=len(mine), nfasc_file=int(nf),
                         dfasc_mean=mine.mean(), dfasc_mean_file=df_.mean(),
                         ct_d=row.get("ct_d", np.nan)))
chk = pd.DataFrame(rows)
chk["species"] = chk["sample"].str[0].map({"C": "human", "P": "pig", "R": "rat"})
chk["dnerve_ratio"] = chk.dnerve_file / chk.dnerve
chk["dfasc_ratio"] = chk.dfasc_mean_file / chk.dfasc_mean
summ = chk.groupby(["species", "version"]).agg(
    n=("sample", "size"), dnerve_exact=("dnerve_ratio", lambda x: int((abs(x - 1) < 1e-4).sum())),
    dnerve_ratio_min=("dnerve_ratio", "min"), dnerve_ratio_max=("dnerve_ratio", "max"),
    nfasc_equal=("nfasc", lambda x: int((x == chk.loc[x.index, "nfasc_file"]).sum())),
    dfasc_ratio_median=("dfasc_ratio", "median"))
print(summ.round(4))
print("\nRat samples whose file nerve diameter is the connective tissue boundary (multifascicular):")
r_ = chk[(chk.species == "rat") & (chk.version == "v5")].merge(tab[["sample", "n_fasc"]], on="sample")
print(r_.loc[(r_.n_fasc > 1) & (abs(r_.dnerve_file - r_.ct_d) < 1e-6), ["sample", "dnerve", "dnerve_file", "ct_d"]].round(1).to_string(index=False))
print("\nPig MT: fascicle counts, these masks (v4) against the file:")
p_ = chk[chk.species == "pig"]
print(p_[["sample", "nfasc", "nfasc_file", "dfasc_mean", "dfasc_mean_file"]].round(1).to_string(index=False))

In [ ]:
# Pig MT fascicle masks in version 1 of sparc-64 (all 18 samples), against version 4 and the file
v1c, v1d = [], []
for s in MAIN[MAIN.species == "pig"]["sample"]:
    sub, sam = s[1:].split("-")
    paths = pelot.browse("sparc-64", 1, f"files/primary/sub-{sub}/sam-{sam}")
    fm = [p for p in paths if p.endswith("FascMask.tif")][0]
    o = pelot.objects(pelot.read_mask("sparc-64", fm, 1), with_holes=True)
    o = o + [h for x in o for h in x["holes"]]
    k = pelot.scale_um_per_px(pelot.read_mask("sparc-64", [p for p in paths if p.endswith("ScaleMask.tif")][0], 1), 500.0)
    d = pelot.eff_d(np.array([x["area"] for x in o]) * k * k)
    v1d.append(d)
    file_d = np.atleast_1d(om["dFasc_um"][list(om["sub_sam"]).index(s)]).astype(float)
    v1c.append(dict(sample=s, scale_v1_over_v4=k / res[s]["um_per_px"], n_v1=len(o), n_v4=len(res[s]["fasc"]), n_file=int(chk.loc[chk["sample"] == s, "nfasc_file"].iloc[0]),
                    mean_d_v1=d.mean(), mean_d_v4=res[s]["fasc"].d_in.mean(), mean_d_file=file_d.mean()))
v1c = pd.DataFrame(v1c)
print(v1c.round(1).to_string(index=False))
print(f"samples where version 1 count = file count: {int((v1c.n_v1 == v1c.n_file).sum())} of 18; version 4 = file: {int((v1c.n_v4 == v1c.n_file).sum())}; "
      f"version 1 = version 4: {int((v1c.n_v1 == v1c.n_v4).sum())}")
d_v1 = np.concatenate(v1d)
print(f"version 1, all {len(d_v1)} traces: {d_v1.min():.1f} to {d_v1.max():.1f} um, mean {d_v1.mean():.1f}, median {np.median(d_v1):.1f}")
print("per sample, mean fascicle diameter in the file over these masks (v4):", (v1c.mean_d_file / v1c.mean_d_v4).round(3).agg(["min", "median", "max"]).to_dict())
eq = [s for s in v1c["sample"] if (v1c.set_index("sample").loc[s, ["n_v1", "n_file"]].nunique() == 1)]
r_ = []
for s in eq:
    file_d = np.sort(np.atleast_1d(om["dFasc_um"][list(om["sub_sam"]).index(s)]).astype(float))
    mine = np.sort(v1d[v1c.index[v1c["sample"] == s][0]])
    r_.append(file_d / mine)
r_ = np.concatenate(r_)
print(f"per fascicle, file over version 1 masks (samples with equal counts, sorted, {len(r_)} fascicles): ratio {r_.min():.3f} to {r_.max():.3f}, median {np.median(r_):.4f}")
# Pig IF: against PigIFMorphology.mat
pm = pelot.read_mat("sparc-89", "files/code/PigIFMorphology.mat")
print("\nPig IF endoneurial % (file, cervical and subdiaphragmatic means):", np.round(pm["pigIF_Aendoperc_mean"], 2),
      " these masks:", IF.assign(e=100 * IF.endo_area / IF.nerve_area).groupby("level").e.mean().round(2).to_list())
print("Pig IF perineurial % (file):", np.round(pm["pigIF_Aperiperc_mean"], 2),
      " these masks:", IF.assign(e=100 * IF.peri_area / IF.nerve_area).groupby("level").e.mean().round(2).to_list())

In [ ]:
# Human nerve traces: sparc-65 holds NerveMask.tif for every sample and, for some, a second
# trace NerveMask_AllTissue.tif. HumanRatSegmentationAnalysis_v2.m reads NerveMask.tif
# (nervefilename); the Figure 2 legend says four human samples needed manual adjustment to
# exclude tissue that was not part of the nerve trunk.
alltis = sorted(p for p in pelot.files("sparc-65") if p.endswith("NerveMask_AllTissue.tif"))
rows = []
for p in alltis:
    m = __import__("re").search(r"sub-(\d+)/sam-(\d+)/", p)
    s = f"C{m.group(1)}-{m.group(2)}"
    if s not in res:
        continue
    o = pelot.objects(pelot.read_mask("sparc-65", p))
    k = res[s]["um_per_px"]
    rows.append(dict(sample=s, d_NerveMask=res[s]["nerve_d"], d_AllTissue=float(pelot.eff_d(o[0]["area"] * k * k)),
                     d_file=float(np.atleast_1d(om["dNerve_um"][list(om["sub_sam"]).index(s)])[0])))
print("human samples with an AllTissue trace:", len(rows))
print(pd.DataFrame(rows).round(1).to_string(index=False))
# Rat traces of version 1 against the file, per fascicle (hole traces measured as bwboundaries traces them)
rr = []
for s in tab_v1["sample"]:
    file_d = np.sort(np.atleast_1d(om["dFasc_um"][list(om["sub_sam"]).index(s)]).astype(float))
    mine = np.sort(res_v1[s]["fasc"].d_in.to_numpy())
    rr.append(dict(sample=s, n=len(mine), max_abs_rel_diff=np.max(np.abs(mine / file_d - 1))))
rr = pd.DataFrame(rr)
print("\nrat version 1 inner diameters against the file, largest relative difference per sample:")
print(rr.assign(max_abs_rel_diff=rr.max_abs_rel_diff.map(lambda x: f"{x:.1e}")).to_string(index=False))

In [ ]:
# Subject metadata (rat) and the readme's statements
subj60 = pelot.read_excel("sparc-60", "files/subjects.xlsx")
print(subj60[["subject_id", "age", "sex", "strain", "body mass", "body mass units"]].to_string(index=False))
readme60 = pelot.read_bytes("sparc-60", "readme.md").decode()
print("\nreadme.md of sparc-60 says:", [l.strip() for l in readme60.splitlines() if "days old" in l])

In [ ]:
# Check of mixed.py on synthetic balanced designs, where Kenward-Roger and Satterthwaite must
# give the classical split-plot F tests and degrees of freedom
rng_chk = np.random.default_rng(0)  # synthetic data for a software check only, no claim
rows = [dict(sp=a, lev=b, subject=f"{a}{s}", sample=f"{a}{s}{b}", y=u + w + rng_chk.normal() + 0.5 * (b == "C"))
        for a in "xyz" for s in range(5) for u in [rng_chk.normal()] for b in "CS" for w in [rng_chk.normal(0, .7)] for r in range(4)]
syn = pd.DataFrame(rows)
fs = mixed.fit(syn, "y", mixed.Design(syn, "sp", "lev", list("xyz"), ["C", "S"]), ["subject", "sample"], bounded=False)
print("Kenward-Roger:"); print(mixed.anova(fs).round(4).to_string(index=False))
print("Satterthwaite:"); print(mixed.anova(fs, "satterthwaite").round(4).to_string(index=False))
cell = syn.groupby("sample").y.mean().to_frame().join(syn.groupby("sample")[["sp", "lev", "subject"]].first())
import statsmodels.formula.api as smf
a = smf.ols("y ~ C(sp, Sum) * C(lev, Sum) + C(subject)", data=cell).fit()
from statsmodels.stats.anova import anova_lm
t = anova_lm(a, typ=1)
F_lev = t.loc["C(lev, Sum)", "mean_sq"] / t.loc["Residual", "mean_sq"]
print(f"classical split plot on the sample means: level F = {F_lev:.4f} on (1, {int(t.loc['Residual', 'df'])}) df")

The data check finds:

- The human measurements reproduce the deposited output metrics exactly (nerve diameter, number of
  fascicles and fascicle diameters of all 18 samples), once masks stored as photometric MinIsWhite
  are inverted as MATLAB's `imread` returns them. The boundary convention of this notebook is
  therefore the code's. The human nerve diameters are those of `NerveMask.tif`, the trace the code
  reads and the file holds; the four samples that also have `NerveMask_AllTissue.tif` (C47-2,
  C54-3, C55-3, C63-1) are the four the Figure 2 legend says were adjusted by hand, and their
  AllTissue traces are larger (1328 to 2692 um).
- Pig MT nerve diameters match the file exactly. The pig fascicle counts of the file are those of
  the version 1 masks of sparc-64 in all 18 samples (version 4 differs in 11). The fascicle
  diameters in the file are about 1.40 times those of the masks of either version, and the ratio
  is not a constant factor (1.36 to 1.61 per fascicle), so it is not a scale; it is unexplained.
  The paper's own pig MT values (mean 135, median 130 um) match the masks, not the file.
- The rat values of the file match the version 1 masks of sparc-60: the cervical inner diameters
  exactly (within 4e-6, with hole traces measured as `bwboundaries` traces them), the
  subdiaphragmatic ones within 0.5%, except R26-1 (11%). The version 5 outer perineurium boundaries
  are smaller (nerve diameter ratios printed above), so the rat perineurium is thinner in version
  5. For the three multifascicular rat samples the file gives the connective tissue boundary as the
  nerve diameter, where the Methods use the outer perineurium of the largest fascicle.
- sparc-60's readme gives the rats as 65 to 268 days old; the subjects sheet gives 75 to 268 days.
- `mixed.py` gives the classical split-plot F test and degrees of freedom on a balanced synthetic
  design, with Kenward-Roger and with Satterthwaite degrees of freedom.

## rat-sample-set (link)

> The rat traces hold the rat samples the paper analysed, segmentations of nine left cervical and
> nine anterior subdiaphragmatic vagus nerve samples from Sprague-Dawley rats, 4F and 5M at each
> level, from 10 rats (5M/5F).

In [ ]:
rat = MAIN[MAIN.species == "rat"]
print(rat.groupby(["level", "sex"]).size().unstack())
print("rats:", rat.subject.nunique(), "; sex of rats:", rat.drop_duplicates("subject").sex.value_counts().to_dict())
smp60 = pelot.read_excel("sparc-60", "files/samples.xlsx")
smp60["key"] = "R" + smp60.subject_id.str[4:] + "-" + smp60.sample_id.str[4:]
loc = dict(zip(smp60.key, smp60["specimen anatomical location"]))
print("level per samples.xlsx agrees with the code's lists:",
      all(("cervical" in loc[s]) == (l == "C") for s, l in zip(rat["sample"], rat.level)))
print("strain:", subj60.strain.unique(), "; ages (days):", subj60.age.str.split().str[0].astype(int).agg(["min", "max"]).to_list())
w = subj60["body mass"].astype(float)
print(f"body mass over {len(w)} rats: {w.mean():.1f} +/- {w.std():.1f} g (SD)")
print("extra samples in sparc-60 beyond the 18:", sorted(set(smp60.key) - set(rat["sample"])))

**Result: reproduced.** sparc-60 holds 9 cervical and 9 subdiaphragmatic rat samples, 4F and 5M at
each level, from 10 Sprague-Dawley rats (5M, 5F), aged 75 to 268 days, 381.1 +/- 118.4 g, as the
Methods state. R18-1 is a cervical sample (R18 has no subdiaphragmatic sample; R26-1 is in its
place). No sample beyond the 18 is in the dataset.

## human-sample-set (link)

> The human anti-claudin-1 data hold the human samples the paper analysed, nine left cervical and
> nine anterior subdiaphragmatic samples, 4F and 5M at each level.

In [ ]:
hum = MAIN[MAIN.species == "human"]
print(hum.groupby(["level", "sex"]).size().unstack())
subj65 = pelot.read_excel("sparc-65", "files/subjects.xlsx")
sx = dict(zip(subj65.subject_id.str[4:].astype(int), subj65.sex.str[0]))
print("sex in subjects.xlsx agrees with the code's lists:", all(sx[s] == x for s, x in zip(hum["sub"], hum.sex)))
named = ["C47-2", "C54-3", "C55-3", "C56-3", "C63-1", "C57-3"]
print("named samples present:", {s: s in set(hum["sample"]) for s in named})
have = {s: all(any(p.startswith(f"files/primary/sub-{s[1:].split('-')[0]}/sam-{s.split('-')[1]}/") and p.endswith(m)
                   for p in pelot.files("sparc-65")) for m in ["NerveMask.tif", "PeriExtMask.tif", "PeriInnerMask.tif"])
        for s in hum["sample"]}
mic = {s: any(p.startswith(f"files/primary/sub-{s[1:].split('-')[0]}/sam-{s.split('-')[1]}/") and p.endswith(".nd2")
              for p in pelot.files("sparc-65")) for s in hum["sample"]}
print("samples with nerve, outer and inner masks:", sum(have.values()), "; with an nd2 micrograph:", sum(mic.values()))
print("human subjects:", hum.subject.nunique(), "; ages (years, code lists; 99 = 90+):", sorted(hum.drop_duplicates('subject').age_code))

**Result: reproduced.** sparc-65 holds anti-claudin-1 micrographs and nerve, outer and inner
perineurium traces of 9 cervical and 9 subdiaphragmatic human samples, 4F and 5M at each level,
including C47-2, C54-3, C55-3, C56-3, C63-1 and C57-3.

## pig-mt-sample-set (link)

> The pig MT traces hold the pig samples the paper analysed, nine left cervical (6F/3M) and nine
> anterior subdiaphragmatic (6F/3M) samples, segmented for nerve and inner perineurium boundaries.

In [ ]:
pig = MAIN[MAIN.species == "pig"]
print(pig.groupby(["level", "sex"]).size().unstack())
subj64 = pelot.read_excel("sparc-64", "files/subjects.xlsx")
sx = dict(zip(subj64.subject_id.str[4:].astype(int), subj64.sex.str[0]))
print("sex in subjects.xlsx agrees:", all(sx[s] == x for s, x in zip(pig["sub"], pig.sex)))
print("pigs:", pig.subject.nunique(), "; subjects in subjects.xlsx:", len(subj64))
print("P11-1 and P13-1 present:", {s: s in set(pig["sample"]) for s in ["P11-1", "P13-1"]})
print("outer perineurium masks in sparc-64:", sum(p.endswith("PeriExtMask.tif") for p in pelot.files("sparc-64")))

**Result: reproduced.** sparc-64 holds nerve and fascicle (inner perineurium) traces of 9 cervical
and 9 subdiaphragmatic pig samples, 6F and 3M at each level, from 11 pigs, including P11-1 and
P13-1; it has no outer perineurium traces, as the paper says.

## pig-if-sample-set (link)

> The pig anti-fibronectin data hold four cervical (2F/2M) and four subdiaphragmatic (2F/2M) pig
> samples segmented for nerve, outer and inner perineurium boundaries.

In [ ]:
ifs = pelot.if_samples()
print(ifs[["sample", "level", "sex", "location"]].to_string(index=False))
print(ifs.groupby(["level", "sex"]).size().unstack())
print("each among the 18 pig MT samples at the same level:",
      all(((pig["sample"] == s) & (pig.level == l)).any() for s, l in zip(ifs["sample"], ifs.level)))
print("P12-1 present:", "P12-1" in set(ifs["sample"]))

**Result: reproduced.** sparc-89 holds nerve, outer and inner perineurium traces of 4 cervical and 4
subdiaphragmatic pig samples, 2F and 2M at each level, each one of the MT samples at the same level,
including P12-1.

## mt-micrograph-collections (link)

> The MT micrograph collections hold cross sections of the samples collected, 37 cervical and 32
> subdiaphragmatic human samples from 30 subjects, 21 cervical and 24 subdiaphragmatic pig samples
> from 12 pigs, and 18 cervical and 9 subdiaphragmatic rat samples from 10 rats.

Counted from each collection's `samples.xlsx` (level from the anatomical location) and from the
micrograph files (`.nd2` in `files/primary/`), and the subjects from `subjects.xlsx`.

In [ ]:
import re
coll = []
for ds, sp in [("sparc-61", "human"), ("sparc-82", "pig"), ("sparc-16", "rat")]:
    smp = pelot.read_excel(ds, "files/samples.xlsx")
    subj = pelot.read_excel(ds, "files/subjects.xlsx")
    locs = smp["specimen anatomical location"].astype(str).str.lower()
    nd2 = [p for p in pelot.files(ds) if p.startswith("files/primary/") and p.endswith(".nd2")]
    keyed = {}
    for p in nd2:
        m = re.match(r"files/primary/(sub-\d+)/(sam-\d+)/", p)
        keyed.setdefault(m.groups(), []).append(p)
    lev = {(a, b): ("C" if "cervical" in l else "S") for a, b, l in zip(smp.subject_id, smp.sample_id, locs)}
    coll.append(dict(dataset=ds, species=sp, samples_cervical=sum(v == "C" for v in lev.values()),
                     samples_subdiaphragmatic=sum(v == "S" for v in lev.values()),
                     subjects=len(subj), sexes=subj.sex.value_counts().to_dict(),
                     samples_with_nd2=len(keyed), nd2_files=len(nd2),
                     nd2_cervical=sum(len(v) for k, v in keyed.items() if lev.get(k) == "C"),
                     nd2_subdiaphragmatic=sum(len(v) for k, v in keyed.items() if lev.get(k) == "S"),
                     nd2_without_sample_row=sorted(set(keyed) - set(lev))))
coll = pd.DataFrame(coll)
print(coll.drop(columns="nd2_without_sample_row").to_string(index=False))
print(coll[["dataset", "nd2_without_sample_row"]].to_string(index=False))
smp61 = pelot.read_excel("sparc-61", "files/samples.xlsx")
no_nd2 = [(a, b, l) for a, b, l in zip(smp61.subject_id, smp61.sample_id, smp61["specimen anatomical location"])
          if not any(p.startswith(f"files/primary/{a}/{b}/") and p.endswith(".nd2") for p in pelot.files("sparc-61"))]
print("sparc-61 sample rows without an nd2 in their folder:", no_nd2)
print("sparc-61 nd2 in sub-48/sam-2:", [p for p in pelot.files("sparc-61") if p.startswith("files/primary/sub-48/") and p.endswith(".nd2")])
for ds in ["sparc-61", "sparc-82", "sparc-16"]:
    smp = pelot.read_excel(ds, "files/samples.xlsx")
    analysed = MAIN[MAIN.species == {"sparc-61": "human", "sparc-82": "pig", "sparc-16": "rat"}[ds]]
    keys = set(zip(smp.subject_id, smp.sample_id))
    print(ds, "analysed samples present:", sum((f"sub-{a}", f"sam-{b}") in keys for a, b in zip(analysed["sub"], analysed.sam)), "of", len(analysed))

**Result: reproduced.** sparc-61 has 37 cervical and 32 subdiaphragmatic human samples from 30
subjects (15F/15M), sparc-82 21 cervical and 24 subdiaphragmatic pig samples from 12 pigs, and
sparc-16 18 cervical and 9 subdiaphragmatic rat samples from 10 rats. Every sample row has an nd2
micrograph in its folder (two for each rat subdiaphragmatic sample, anterior and posterior) except
one in sparc-61: the right cervical sample sub-48/sam-1 has no folder, and the micrograph named
C48-2 is in sub-48/sam-2, which has no row; counted once, it makes 37 human cervical micrographs.
Every analysed sample of each species is in its collection.

## nerve-diameter-anova

> Effective nerve diameter differs across species and across nerve levels, with no detected
> interaction of species and level.

Mixed model of effective nerve diameter (um), fixed species, level and interaction, random subject,
REML; readings (a) variance component unbounded (JMP default), (b) bounded at zero. Rat nerve
diameter is the outer perineurium of the largest fascicle (Methods). Denominator degrees of freedom
by Kenward-Roger, the claims file's (JMP's) method; Satterthwaite is shown as a variant.

In [ ]:
D3 = mixed.Design(MAIN, "species", "level", ["human", "pig", "rat"], ["C", "S"])
nd_fit = {}
for reading, bounded in [("a unbounded", False), ("b bounded", True)]:
    f = mixed.fit(MAIN, "nerve_d", D3, ["subject"], bounded=bounded)
    nd_fit[reading] = f
    print(f"reading {reading}: residual variance {f.sigma2:.0f}, subject variance {f.tau2['subject']:.0f} um^2")
    print("  Kenward-Roger (JMP's method):"); print(mixed.anova(f).round(4).to_string(index=False))
    print("  variant, Satterthwaite:"); print(mixed.anova(f, "satterthwaite").round(4).to_string(index=False))
print("\npaper: species F = 577.1, p < 0.0001; level F = 11.01, p = 0.0030; interaction F = 2.445, p = 0.1085")
# variant: rat version 1 traces
MAIN_v1 = pd.concat([MAIN[MAIN.species != "rat"], tab_v1])
f = mixed.fit(MAIN_v1, "nerve_d", mixed.Design(MAIN_v1, "species", "level", ["human", "pig", "rat"], ["C", "S"]), ["subject"], bounded=False)
print("\nvariant, rat traces of sparc-60 version 1, reading a, Kenward-Roger:")
print(mixed.anova(f).round(4).to_string(index=False))

**Result: reproduced.** Under reading (a), with Kenward-Roger degrees of freedom, species
F = 587.3 (p < 0.0001), level F = 11.03 (p = 0.0023) and interaction F = 2.45 (p = 0.103), against
the paper's 577.1, 11.01 (p = 0.0030) and 2.445 (p = 0.1085). Satterthwaite degrees of freedom give
599.6, 11.05 (p = 0.0029) and 2.45 (p = 0.108). Under reading (b) the subject variance is at zero
and the interaction is significant (F = 3.60, p = 0.035), so the absence of an interaction holds
only under JMP's default, reading (a), which the paper's F values match; one reading is enough
under the criterion.

## human-pig-nerve-diameter-similar

> Human and pig vagus nerves have similar effective diameters, 1.9 +/- 0.5 against 2.2 +/- 0.2 mm
> cervically and 1.7 +/- 0.3 against 1.6 +/- 0.2 mm subdiaphragmatically, with no significant
> difference by Tukey's test.

In [ ]:
g = MAIN.groupby(["species", "level"]).nerve_d.agg(["mean", "std", "min", "max", "count"]) / [1000, 1000, 1000, 1000, 1]
print("effective nerve diameter (mm):"); print(g.round(3))
for reading, f in nd_fit.items():
    for method in ["kr", "satterthwaite"]:
        tk = mixed.tukey(f, "a", method=method)
        print(f"\nTukey on species least squares means, reading {reading}, {method}:")
        print(tk[["first", "second", "diff", "se", "df", "p"]].round(4).to_string(index=False))
f = nd_fit["a unbounded"]
tk = mixed.tukey(f, "ab")
sel = tk[[a[0] == "human" and b[0] == "pig" and a[1] == b[1] for a, b in zip(tk["first"], tk["second"])]]
print("\nvariant, human against pig at each level (species by level Tukey, reading a, Kenward-Roger):")
print(sel[["first", "second", "diff", "df", "p"]].round(4).to_string(index=False))
hp = MAIN[MAIN.species.isin(["human", "pig"])]
for lev in ["C", "S"]:
    a, b = hp[(hp.species == "human") & (hp.level == lev)].nerve_d, hp[(hp.species == "pig") & (hp.level == lev)].nerve_d
    print(f"{LEVEL[lev]}: human - pig = {(a.mean() - b.mean()) / 1000:.3f} mm, difference over the mean SD = {(a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2):.2f}")

**Result: reproduced.** Means are 1.92 +/- 0.46 (human) and 2.20 +/- 0.19 mm (pig) cervically and
1.68 +/- 0.32 and 1.64 +/- 0.16 mm subdiaphragmatically, all within the paper's SDs. Human against
pig, species main-effect contrast, p = 0.159 under reading (a) with Kenward-Roger df (paper
p = 0.1830; 0.157 with Satterthwaite df) and p = 0.345 under (b); at each level, p = 0.138
cervically and p = 0.988 subdiaphragmatically (variant).

## rat-nerve-diameter

> Rat vagus nerves have effective diameters of 0.260 +/- 0.025 mm at the cervical level and
> 0.135 +/- 0.015 mm at the subdiaphragmatic level.

In [ ]:
rat = MAIN[MAIN.species == "rat"]
print("rat effective nerve diameter (um), outer perineurium of the largest fascicle, version 5:")
print(rat.groupby("level").nerve_d.agg(["mean", "std", "min", "max", "count"]).round(2))
print("largest fascicle by outer area is also the largest by inner area in every sample:", bool(rat.largest_by_inner_same.all()))
print("\nvariant, version 1 traces:")
print(tab_v1.groupby("level").nerve_d.agg(["mean", "std"]).round(2))
ct = rat.assign(d=np.where(rat.n_fasc > 1, rat.ct_d, rat.nerve_d))
print("\nvariant, connective tissue boundary for the three multifascicular samples:")
print(ct.groupby("level").d.agg(["mean", "std"]).round(2))

**Result: reproduced.** Cervical 257.7 +/- 24.9 um and subdiaphragmatic 135.2 +/- 14.8 um, against
260 +/- 25 and 135 +/- 15; means and SDs are within tolerance. The version 1 traces give 261.2 and
137.8 um. With the connective tissue boundary of the three multifascicular
samples, the subdiaphragmatic mean is 174.7 um, outside the tolerance, so the paper's value uses the
largest fascicle.

## rat-ten-times-smaller

> Rat vagus nerves are about ten times smaller in diameter than human and pig nerves, significantly
> smaller than each of the two larger species.

In [ ]:
for reading, f in nd_fit.items():
    tk = mixed.tukey(f, "a")
    print(f"reading {reading}, Kenward-Roger:"); print(tk[["first", "second", "diff", "df", "p"]].to_string(index=False))
f = nd_fit["a unbounded"]; tk = mixed.tukey(f, "ab")
sel = tk[[(a[0] == "rat") != (b[0] == "rat") and a[1] == b[1] for a, b in zip(tk["first"], tk["second"])]]
print("\nvariant, per level (reading a):"); print(sel[["first", "second", "diff", "p"]].to_string(index=False))
m = MAIN.groupby(["species", "level"]).nerve_d.mean()
for lev in ["C", "S"]:
    print(f"{LEVEL[lev]}: human/rat = {m['human', lev] / m['rat', lev]:.1f}, pig/rat = {m['pig', lev] / m['rat', lev]:.1f}")

**Result: reproduced.** Rat against human and against pig, Tukey on the species means, p < 1e-12
under both readings (rat smaller), and the larger-species over rat ratios are 7.5 and 8.5
cervically and 12.4 and 12.1 subdiaphragmatically, all between 5 and 20.

## nerve-cervical-larger

> Cervical vagus nerves are larger than subdiaphragmatic nerves, a trend clearest in pigs and rats.

In [ ]:
f = nd_fit["a unbounded"]
print(mixed.anova(f).round(4).iloc[[1]].to_string(index=False))
print("level least squares means (um):", [(x["b"], round(x["lsmean"], 1)) for x in mixed.lsmeans(f, "b")])
fb = nd_fit["b bounded"]
print("reading b level effect:", mixed.anova(fb).round(5).iloc[[1]][["F", "p"]].to_dict("records"))
rows = []
for sp in ["human", "pig", "rat"]:
    c, s = [MAIN[(MAIN.species == sp) & (MAIN.level == l)].nerve_d for l in "CS"]
    pooled = np.sqrt(((len(c) - 1) * c.var() + (len(s) - 1) * s.var()) / (len(c) + len(s) - 2))
    rows.append(dict(species=sp, cervical=c.mean(), subdiaphragmatic=s.mean(),
                     a_relative_difference=(c.mean() - s.mean()) / c.mean(), b_difference_over_pooled_sd=(c.mean() - s.mean()) / pooled))
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: reproduced.** Level F = 11.03, p = 0.0023 (Kenward-Roger), with the cervical least
squares mean larger (1425 against 1131 um). The relative difference is 0.13 for humans, 0.25 for
pigs and 0.48 for rats (reading a), and the difference over the pooled SD 0.62, 3.11 and 5.97
(reading b), so the trend is clearer in pigs and rats under both readings; the outcome of that
part rests on these readings of "clearest".

## fascicle-number-human-pig

> Pig vagus nerves have almost ten times more fascicles than human nerves (47 +/- 10 and 38 +/- 7
> against 6.7 +/- 4.5 and 4.9 +/- 4.0 at the cervical and subdiaphragmatic levels), the number
> differs by species and level with no detected interaction, and there are more fascicles
> cervically.

In [ ]:
hp = MAIN[MAIN.species.isin(["human", "pig"])].copy()
print(hp.groupby(["species", "level"]).n_fasc.agg(["mean", "std", "min", "max", "count"]).round(2))
D2 = mixed.Design(hp, "species", "level", ["human", "pig"], ["C", "S"])
for reading, bounded in [("a unbounded", False), ("b bounded", True)]:
    f = mixed.fit(hp, "n_fasc", D2, ["subject"], bounded=bounded)
    print(f"\nreading {reading}: subject variance {f.tau2['subject']:.2f}, residual {f.sigma2:.2f}")
    print("  Kenward-Roger:"); print(mixed.anova(f).round(4).to_string(index=False))
    print("  variant, Satterthwaite:"); print(mixed.anova(f, "satterthwaite").round(4).to_string(index=False))
    print("level least squares means:", [(x["b"], round(x["lsmean"], 2)) for x in mixed.lsmeans(f, "b")])
print("\npaper: species F = 213.8, p < 0.0001; level F = 5.486, p = 0.0300; interaction F = 2.481, p = 0.1322")

**Result: reproduced.** Means are 46.9 +/- 10.3 and 38.2 +/- 7.1 fascicles for pigs and 6.7 +/- 4.5
and 4.9 +/- 4.0 for humans, all within the paper's SDs. With Kenward-Roger df, species F = 212.9
(p < 0.0001), level F = 5.39 (p = 0.033) with the cervical least squares mean higher (26.4 against
21.7), and interaction F = 2.43 (p = 0.137), against the paper's 213.8, 5.486 (p = 0.0300) and 2.481
(p = 0.1322); the same under both readings (the subject variance is positive). Satterthwaite df give
216.7, 5.79 (p = 0.026) and 2.61 (p = 0.12).

## rat-subdiaphragmatic-multifascicular

> Rat vagus nerves are generally monofascicular; three of the nine rat subdiaphragmatic samples
> (R16-1, R22-1 and R23-1) are multifascicular.

In [ ]:
print(rat[["sample", "level", "n_fasc"]].sort_values(["level", "sample"]).to_string(index=False))
print("multifascicular:", rat[rat.n_fasc > 1]["sample"].to_list())
print("version 1:", tab_v1[tab_v1.n_fasc > 1][["sample", "n_fasc"]].values.tolist())

**Result: reproduced.** All nine cervical samples have one fascicle; R16-1 (2), R22-1 (3) and
R23-1 (2) are the only multifascicular subdiaphragmatic samples.

## rat-multifascicular-proportions

> Counting all segmented fascicles and the surrounding tissue, the epineurium, perineurium and
> endoneurium proportions are 58.0, 3.8 and 38.2% for R16-1, 55.1, 4.2 and 40.7% for R23-1, and
> 63.4, 3.6 and 33.0% for R22-1.

In [ ]:
exp = {"R16-1": (58.0, 3.8, 38.2), "R23-1": (55.1, 4.2, 40.7), "R22-1": (63.4, 3.6, 33.0)}
rows = []
for ver, t in [("v5", tab), ("v1", tab_v1)]:
    for s, e in exp.items():
        r = t[t["sample"] == s].iloc[0]
        endo, peri = 100 * r.endo_area / r.ct_area, 100 * r.peri_area / r.ct_area
        got = (100 - endo - peri, peri, endo)
        rows.append(dict(version=ver, sample=s, epi=got[0], peri=got[1], endo=got[2],
                         within_10pct=sum(abs(g - x) <= 0.1 * x for g, x in zip(got, e))))
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: reproduced.** With the version 5 traces all 9 values are within 10%: R16-1 57.95, 3.82
and 38.23%, R23-1 55.15, 4.19 and 40.66%, R22-1 63.42, 3.61 and 32.96% (epineurium, perineurium,
endoneurium). With the version 1 traces the perineurium proportions (4.82, 6.59, 4.71%) are above
the paper's, and 6 of 9 values hold.

## fascicle-diameter-anova

> Effective fascicle diameter differs across species and nerve levels, with a significant species
> by level interaction.

log10 of every fascicle's effective diameter (inner perineurium), random subject and sample,
Kenward-Roger df (Satterthwaite as a variant); rats contribute the largest fascicle per sample, as
the criterion says. The paper's code keeps a different rat fascicle: DataPlotting_v4.m sorts the
outer perineurium diameters of each rat sample (for the nerve diameter) but keeps `FascicleIn{1}`,
the inner trace of the first ring in `bwboundaries` order, as the fascicle diameter. That
selection is shown as a variant here and in the claims that use this model.

In [ ]:
fr = []
for sp in ["human", "pig", "rat"]:
    t = pelot.fascicles(res, MAIN, sp)
    if sp == "rat":
        t = t.loc[t.groupby("sample").a_out.idxmax()]
    t["species"] = sp
    fr.append(t)
FASC = pd.concat(fr, ignore_index=True)
FASC["log_d"] = np.log10(FASC.d_in)
print(FASC.groupby(["species", "level"]).d_in.agg(["count", "mean", "std", "min", "max"]).round(1))
DF3 = mixed.Design(FASC, "species", "level", ["human", "pig", "rat"], ["C", "S"])
fd_fit = {}
for reading, bounded in [("a unbounded", False), ("b bounded", True)]:
    f = mixed.fit(FASC, "log_d", DF3, ["subject", "sample"], bounded=bounded)
    fd_fit[reading] = f
    print(f"\nreading {reading}: variances subject {f.tau2['subject']:.5f}, sample {f.tau2['sample']:.5f}, residual {f.sigma2:.5f}")
    print("  Kenward-Roger:"); print(mixed.anova(f).round(4).to_string(index=False))
    print("  variant, Satterthwaite:"); print(mixed.anova(f, "satterthwaite").round(4).to_string(index=False))
print("\npaper: species F = 8.989, p = 0.0015; level F = 51.88, p < 0.0001; interaction F = 5.273, p = 0.0121")
# variant: the rat fascicle the paper's code keeps. DataPlotting_v4.m sorts the rat outer
# diameters (FascicleOut, for the nerve diameter) but keeps FascicleIn{1}, the inner trace of
# the first ring in bwboundaries order (column-major bwlabel order), as the fascicle diameter
FASC_first = pd.concat([FASC[FASC.species != "rat"],
                        pelot.fascicles(res, MAIN, "rat").query("outer == 0").assign(species="rat")], ignore_index=True)
FASC_first["log_d"] = np.log10(FASC_first.d_in)
fd_first = mixed.fit(FASC_first, "log_d", mixed.Design(FASC_first, "species", "level", ["human", "pig", "rat"], ["C", "S"]),
                     ["subject", "sample"], bounded=False)
print("\nvariant, rat first fascicle as the code keeps it (reading a, Kenward-Roger):")
print(mixed.anova(fd_first).round(4).to_string(index=False))

**Result: reproduced.** With Kenward-Roger df, species F = 9.66 (p = 0.0004), level F = 47.3
(p < 0.0001) and interaction F = 5.20 (p = 0.0088), against 8.989, 51.88 and 5.273 in the paper;
Satterthwaite df give 9.74 (p = 0.0011), 48.0 and 5.46 (p = 0.0099). Both readings give the same
fit, as no variance component is negative. With the rat fascicle the code keeps (variant), F = 9.05,
52.8 and 5.54, closer to the paper's.

## rat-fascicle-diameter

> The largest rat fascicle has an effective diameter of 0.249 +/- 0.024 mm at the cervical level and
> 0.121 +/- 0.032 mm at the subdiaphragmatic level.

In [ ]:
print("largest fascicle, inner perineurium effective diameter (um), version 5:")
print(rat.groupby("level").fasc_d_largest.agg(["mean", "std", "min", "max"]).round(2))
print("geometric mean:", rat.groupby("level").fasc_d_largest.apply(lambda x: 10 ** np.log10(x).mean()).round(2).to_dict())
print("version 1:", tab_v1.groupby("level").fasc_d_largest.agg(["mean", "std"]).round(2).to_dict())
rf = pelot.fascicles(res, MAIN, "rat")
print("\nsubdiaphragmatic rat fascicles in bwlabel order (outer = ring index; the first ring is outer 0):")
print(rf[rf.level == "S"][["sample", "outer", "d_in", "d_out"]].round(1).to_string(index=False))
first = rf[rf.outer == 0]
print("\nvariant, the first ring's inner trace, as DataPlotting_v4.m keeps it (um):")
print(first.groupby("level").d_in.agg(["mean", "std"]).round(1))
print("samples where the first ring is not the largest:", sorted(set(first["sample"]) - set(rf.loc[rf.groupby("sample").a_out.idxmax(), "sample"][rf.loc[rf.groupby("sample").a_out.idxmax()].outer == 0])))

**Result: reproduced.** The largest fascicle measures 248.6 +/- 24.3 um cervically and
129.1 +/- 14.0 um subdiaphragmatically, both within the paper's SDs of 249 and 121 um; the geometric
means and the version 1 traces give values within the same tolerances. The fascicle the paper's code
keeps, the first ring in `bwboundaries` order, gives 248.6 +/- 24.3 and 121.1 +/- 32.3 um, the
paper's 0.249 +/- 0.024 and 0.121 +/- 0.032 mm. The two choices differ in R22-1 and R23-1: in
R22-1 the first ring is a 39.3 um fascicle and the largest is 105.8 um. So the paper's rat
fascicle diameters follow the code, not the Methods' "largest fascicle".

## cervical-fascicle-diameter-species

> At the cervical level human fascicles (0.355 +/- 0.226 mm) are larger than pig fascicles
> (0.155 +/- 0.058 mm), while rat fascicles are comparable to both.

In [ ]:
f = fd_fit["a unbounded"]
TK_FD = mixed.tukey(f, "ab")
print("Kenward-Roger:"); print(TK_FD[["first", "second", "diff", "se", "df", "p"]].round(4).to_string(index=False))
TK_S = mixed.tukey(f, "ab", method="satterthwaite")
TK_F = mixed.tukey(fd_first, "ab")
cc = lambda t: t[[a[1] == "C" and b[1] == "C" for a, b in zip(t["first"], t["second"])]][["first", "second", "p"]]
print("\ncervical pairs, variants: Satterthwaite; rat first fascicle as the code keeps it (KR)")
print(cc(TK_S).merge(cc(TK_F), on=["first", "second"], suffixes=("_satterthwaite", "_rat_first")).round(4).to_string(index=False))
for sp in ["human", "pig"]:
    t = FASC[(FASC.species == sp)]
    a = t.groupby("level").d_in.agg(["mean", "std"])
    b = t.groupby(["level", "sample"]).d_in.mean().groupby("level").agg(["mean", "std"])
    print(f"{sp}: (a) mean over fascicles {a.round(1).to_dict('index')}; (b) mean of per-sample means {b.round(1).to_dict('index')}")

**Result: reproduced.** Cervically, with Kenward-Roger df, human against pig p = 0.0001 (human
larger; paper 0.0002), rat against human p = 0.72 (paper 0.727) and rat against pig p = 0.13
(paper 0.1707). Satterthwaite df give 0.0002, 0.72 and 0.14, and the code's rat fascicle 0.0001,
0.72 and 0.14. Means over fascicles are 354.8 +/- 226.0 um (human) and 154.5 +/- 57.9 um (pig),
matching the paper's to the micron (reading a); under reading (b), 419.1 and 155.6 um, also within
the SDs.

## subdiaphragmatic-fascicle-diameter-comparable

> At the subdiaphragmatic level fascicle diameters are comparable across species (human 0.163 +/-
> 0.146, pig 0.110 +/- 0.044, rat 0.121 +/- 0.032 mm), although human fascicles trend larger than pig
> fascicles.

In [ ]:
sd_ = lambda t: t[[a[1] == "S" and b[1] == "S" for a, b in zip(t["first"], t["second"])]]
sel = sd_(TK_FD)
print("Kenward-Roger:"); print(sel[["first", "second", "diff", "df", "p"]].round(4).to_string(index=False))
print("\nvariants: Satterthwaite; rat first fascicle as the code keeps it (KR)")
print(sd_(TK_S)[["first", "second", "p"]].merge(sd_(TK_F)[["first", "second", "p"]], on=["first", "second"], suffixes=("_satterthwaite", "_rat_first")).round(4).to_string(index=False))
print("ratio of geometric means for each pair:", {f"{a[0]}/{b[0]}": round(10 ** d, 3) for a, b, d in zip(sel["first"], sel["second"], sel["diff"])})
print(FASC[FASC.level == "S"].groupby("species").d_in.agg(["mean", "std"]).round(1))

**Result: reproduced.** No subdiaphragmatic pair differs: with Kenward-Roger df human against pig
p = 0.28, human against rat p = 1.00, pig against rat p = 0.88 (the paper gives 0.3049, 0.896 and
0.9947). With the rat fascicle the code keeps, the rat p values are 0.898 and 0.995, the paper's to
the third decimal. The means are 163.5 +/- 145.7 um (human) and 110.1 +/- 43.8 um (pig), within
the stated SDs, the human mean above the pig mean.

## fascicles-larger-cervical

> All three species have larger fascicles at the cervical than at the subdiaphragmatic level.

In [ ]:
within = lambda t: t[[a[0] == b[0] for a, b in zip(t["first"], t["second"])]]
print("Kenward-Roger (JMP's method):"); print(within(TK_FD)[["first", "second", "diff", "df", "p"]].round(5).to_string(index=False))
print("\nvariant, Satterthwaite df:"); print(within(TK_S)[["first", "second", "diff", "df", "p"]].round(5).to_string(index=False))
n_sub, n_sam = FASC.subject.nunique(), FASC["sample"].nunique()
df_cont = n_sam - n_sub - 3
tk2 = mixed.tukey(fd_fit["a unbounded"], "ab", df=df_cont)
print(f"\nvariant, containment df for the within-subject stratum ({n_sam} samples - {n_sub} subjects - 3 = {df_cont}):")
print(within(tk2)[["first", "second", "p"]].round(5).to_string(index=False))
print("\nvariant, rat first fascicle as the code keeps it (KR):")
print(within(TK_F)[["first", "second", "diff", "df", "p"]].round(5).to_string(index=False))
print("\npaper: humans p < 0.0001, pigs p = 0.0009, rats p = 0.0234")

**Result: reproduced.** Cervical fascicles are larger in all three species with Kenward-Roger
df, the claims file's method: humans p < 0.00001, pigs p = 0.012 (df 7.6) and rats p = 0.044 (the
paper gives p < 0.0001, 0.0009 and 0.0234). Satterthwaite df give 0.00001, 0.048 and 0.044, and the
rat fascicle the code keeps gives rats p = 0.0089 and pigs p = 0.013. With containment df (16), a
variant, the rat contrast is not significant (p = 0.089). The pig and rat p values are larger than
the paper's under every df method run here, and the rat one is close to 0.05 with the largest
fascicle.

## rat-diameter-like-fascicle

> The monofascicular rat nerves have diameters close to that of a single human or pig fascicle.

In [ ]:
for lev in ["C", "S"]:
    rd = MAIN[(MAIN.species == "rat") & (MAIN.level == lev)].nerve_d.mean()
    hm = FASC[(FASC.species == "human") & (FASC.level == lev)].d_in
    pm_ = FASC[(FASC.species == "pig") & (FASC.level == lev)].d_in
    lo, hi = sorted([hm.mean(), pm_.mean()])
    rng = (min(hm.min(), pm_.min()), max(hm.max(), pm_.max()))
    print(f"{LEVEL[lev]}: rat nerve {rd:.1f} um; human and pig mean fascicle {hm.mean():.1f}, {pm_.mean():.1f}; "
          f"(a) between means {lo <= rd <= hi}; (b) within range {rng[0]:.1f} to {rng[1]:.1f}: {rng[0] <= rd <= rng[1]}")

**Result: reproduced.** The rat mean nerve diameter, 257.7 um cervically and 135.2 um
subdiaphragmatically, lies between the pig and human mean fascicle diameters (154.5 and 354.8 um;
110.1 and 163.5 um) at both levels (reading a), and within the range of single fascicles (reading
b).

## human-pig-tissue-proportions

> Endoneurium is a larger share of the human nerve cervically (32 +/- 7.9%) than subdiaphragmatically
> (8.8 +/- 5.3%), while in pigs it is more even across levels (27 +/- 5.3% and 20 +/- 4.0%, MT),
> slightly below the human cervical share and above the human subdiaphragmatic one; perineurium
> follows the same trends (human 8.9 +/- 2.5% and 3.4 +/- 1.6%, pig 5.1 +/- 1.8% and 4.0 +/- 0.5%).

In [ ]:
P = tab.copy()
P["endo_pct"] = 100 * P.endo_area / P.nerve_area
P["peri_pct"] = 100 * P.peri_area / P.nerve_area
P["epi_pct"] = 100 - P.endo_pct - P.peri_pct.fillna(0)
tp = P[P.species.isin(["human", "pig", "pig_if"])].groupby(["species", "level"])[["endo_pct", "peri_pct", "epi_pct"]].agg(["mean", "std"]).round(2)
print(tp)
exp = [("human", "C", "endo_pct", 32, 7.9), ("human", "S", "endo_pct", 8.8, 5.3), ("pig", "C", "endo_pct", 27, 5.3),
       ("pig", "S", "endo_pct", 20, 4.0), ("human", "C", "peri_pct", 8.9, 2.5), ("human", "S", "peri_pct", 3.4, 1.6),
       ("pig_if", "C", "peri_pct", 5.1, 1.8), ("pig_if", "S", "peri_pct", 4.0, 0.5)]
for sp, lev, col, v, tol in exp:
    m = P[(P.species == sp) & (P.level == lev)][col].mean()
    print(f"{sp:7s} {LEVEL[lev]:16s} {col}: {m:6.2f} (paper {v} +/- {tol}) within: {abs(m - v) <= tol}")
pe = P[P.species == "pig"].groupby("level").endo_pct.mean(); he = P[P.species == "human"].groupby("level").endo_pct.mean()
print("pig MT endoneurium below human cervically:", pe["C"] < he["C"], "; above human subdiaphragmatically:", pe["S"] > he["S"])

**Result: reproduced.** Human endoneurium 32.2 +/- 7.9% and 8.8 +/- 5.3%, pig MT 26.6 +/- 5.3% and
20.3 +/- 4.0%; human perineurium 8.9 +/- 2.5% and 3.4 +/- 1.6%, pig (IF) 5.1 +/- 1.8% and
4.0 +/- 0.5%. Every value is within the paper's SD, and the pig MT endoneurial mean is below the
human one cervically and above it subdiaphragmatically.

## pig-if-endoneurium-lower

> The pig endoneurial proportions from the anti-fibronectin IF micrographs are about 2 percentage
> points lower than those from MT, as the IF segmentation accounts for the perineurium.

In [ ]:
pmt = P[P.species == "pig"].set_index("sample"); pif = P[P.species == "pig_if"].set_index("sample")
for lev in ["C", "S"]:
    a = pmt[pmt.level == lev].endo_pct.mean(); b = pif[pif.level == lev].endo_pct.mean()
    same = pif[pif.level == lev].index
    a2 = pmt.loc[same].endo_pct.mean()
    print(f"{LEVEL[lev]}: MT (9) {a:.2f} - IF (4) {b:.2f} = {a - b:.2f} points; same 4 samples MT {a2:.2f} - IF {b:.2f} = {a2 - b:.2f}")

**Result: reproduced.** The MT mean of nine samples exceeds the IF mean of four by 1.74 points
cervically and 2.21 points subdiaphragmatically, both within 1.5 to 2.5 (the comparison the
sentence makes). Comparing the same four samples, the differences are 3.48 and 5.37 points, outside
the tolerance, so the second reading does not hold.

## rat-level-ratios

> In rats the ratio of mean subdiaphragmatic to mean cervical endoneurial area is 0.32 and the ratio
> of mean nerve diameters is 0.68.

In [ ]:
for ver, t in [("v5", tab[tab.species == "rat"]), ("v1", tab_v1)]:
    c, s = t[t.level == "C"], t[t.level == "S"]
    ra_endo = s.endo_area_largest.mean() / c.endo_area_largest.mean()
    ra_d = s.nerve_d.mean() / c.nerve_d.mean()
    s_ct = np.where(s.n_fasc > 1, s.ct_d, s.nerve_d)
    rb_endo = s.endo_area.mean() / c.endo_area.mean()
    rb_d = s_ct.mean() / c.nerve_d.mean()
    print(f"{ver}: reading (a) largest fascicle: endoneurial area {ra_endo:.3f}, diameter {ra_d:.3f};"
          f" reading (b) all fascicles and connective tissue boundary: endoneurial area {rb_endo:.3f}, diameter {rb_d:.3f}")

**Result: reproduced.** Under reading (b), all fascicles for the endoneurial area and the
connective tissue boundary for the nerve diameter of the three multifascicular samples, the ratios
are 0.317 and 0.678 (paper 0.32 and 0.68). Under reading (a) they are 0.270 and 0.525, both outside
10%. The 0.68 of the Discussion therefore uses the connective tissue boundary, as the deposited
output file does, unlike the Results' means (ratio 0.52).

## human-pig-level-ratios

> The ratio of mean subdiaphragmatic to mean cervical endoneurial area is 0.20 in humans and 0.42
> in pigs, and the ratio of mean nerve diameters is 0.87 and 0.75.

In [ ]:
g = MAIN.groupby(["species", "level"])[["endo_area", "nerve_d"]].mean()
for sp, e1, e2 in [("human", 0.20, 0.87), ("pig", 0.42, 0.75)]:
    r1 = g.loc[(sp, "S"), "endo_area"] / g.loc[(sp, "C"), "endo_area"]
    r2 = g.loc[(sp, "S"), "nerve_d"] / g.loc[(sp, "C"), "nerve_d"]
    print(f"{sp}: endoneurial area ratio {r1:.3f} (paper {e1}), nerve diameter ratio {r2:.3f} (paper {e2})")

**Result: reproduced.** Endoneurial area ratios 0.198 (human) and 0.421 (pig), nerve diameter
ratios 0.873 and 0.746, all within 10% of 0.20, 0.42, 0.87 and 0.75.

## human-variability

> Nerve and fascicle diameters are more consistent across samples in pigs (and rats) than in
> humans.

In [ ]:
rows = []
for sp in ["human", "pig", "rat"]:
    for lev in ["C", "S"]:
        t = MAIN[(MAIN.species == sp) & (MAIN.level == lev)]
        fa = pelot.fascicles(res, MAIN, sp); fa = fa[fa.level == lev]
        if sp == "rat":
            per_sample_mean = t.set_index("sample").fasc_d_largest
            per_sample_range = None
        else:
            per_sample_mean = fa.groupby("sample").d_in.mean()
            per_sample_range = fa.groupby("sample").d_in.agg(lambda x: x.max() - x.min())
        rows.append(dict(species=sp, level=lev, cv_nerve_d=t.nerve_d.std() / t.nerve_d.mean(),
                         cv_mean_fasc_d=per_sample_mean.std() / per_sample_mean.mean(),
                         cv_fasc_range=None if per_sample_range is None else per_sample_range.std() / per_sample_range.mean()))
cv = pd.DataFrame(rows)
print(cv.round(3).to_string(index=False))
for lev in ["C", "S"]:
    c = cv[cv.level == lev].set_index("species")
    print(LEVEL[lev], "human > pig and > rat: nerve", c.cv_nerve_d["human"] > c.cv_nerve_d[["pig", "rat"]].max(),
          "; mean fascicle", c.cv_mean_fasc_d["human"] > c.cv_mean_fasc_d[["pig", "rat"]].max(),
          "; fascicle range (human > pig)", c.cv_fasc_range["human"] > c.cv_fasc_range["pig"])

**Result: reproduced.** The coefficient of variation across samples is largest in humans at both
levels for nerve diameter (0.24 and 0.19 against at most 0.10 and 0.11) and for per-sample mean
fascicle diameter (0.31 and 0.66 against at most 0.10 and 0.14), and the per-sample fascicle range
varies more in humans than in pigs (0.50 and 0.85 against 0.17 and 0.19, reading b). The outcome rests on these readings of "more
consistent".

## pig-oblong-human-round

> Pig vagus nerves are consistently oblong in cross section, whereas human nerves are more circular
> at the cervical level.

In [ ]:
sh = MAIN[MAIN.species.isin(["human", "pig"])][["species", "level", "sample", "aspect", "circularity"]]
print(sh.groupby(["species", "level"])[["aspect", "circularity"]].agg(["median", "min", "max"]).round(3))
hc = sh[(sh.species == "human") & (sh.level == "C")]
pg = sh[sh.species == "pig"]
for col, more_round in [("aspect", "lower"), ("circularity", "higher")]:
    med = hc[col].median()
    if col == "aspect":
        every = (pg[col] > med).all(); medians = all(hc[col].median() < pg[pg.level == l][col].median() for l in "CS")
    else:
        every = (pg[col] < med).all(); medians = all(hc[col].median() > pg[pg.level == l][col].median() for l in "CS")
    print(f"{col}: every pig sample less circular than the human cervical median ({med:.3f}): {every}; human cervical median more circular than the pig median at each level: {medians}")

**Result: reproduced.** Every pig nerve trace is less circular than the median human cervical one
under both readings: aspect ratios of 2.13 to 4.29 against a human cervical median of 1.54, and
circularity of 0.32 to 0.59 against 0.65. The human subdiaphragmatic nerves are as oblong as the
pig ones (median aspect ratio 2.56). The outcome rests on these measures, which the paper does not
name.

## Perineurium thickness against fascicle diameter

Non-peanut fascicles only (Figure 9 legend), both levels pooled, thickness half the difference of
the outer and inner effective diameters. Human from sparc-65, pig from the IF traces of sparc-89,
rat from sparc-60 (all 22 fascicles with both traces; DataPlotting_v4.m fits every rat fascicle with
a thickness, including the small fascicles of the multifascicular samples). The fit is MATLAB's
`fit` with `poly1` (DataPlotting_v4.m and SegmentationAnalysis_v2.m), a straight line with an
intercept, R2 about the mean; the zero-intercept line is the second reading.

In [ ]:
def peri_table(r, t, species):
    x = pelot.fascicles(r, t, species)
    return x[(~x.peanut) & x.thk.notna()].copy()
PERI = {"human": peri_table(res, tab, "human"), "pig": peri_table(res, tab, "pig_if"),
        "rat": peri_table(res, tab, "rat"), "rat v1": peri_table(res_v1, tab_v1, "rat")}
FITS = {}
for sp, t in PERI.items():
    for form, zero in [("i intercept", False), ("ii zero", True)]:
        FITS[sp, form] = pelot.line_fit(t.d_in, t.thk, zero=zero)
    print(f"{sp:7s} n = {len(t):3d}; d_fasc {t.d_in.min():.1f} to {t.d_in.max():.1f} um; thickness {t.thk.min():.2f} to {t.thk.max():.2f} um")
fits = pd.DataFrame([dict(species=k[0], form=k[1], n=v["n"], slope=v["slope"], intercept=v["intercept"], r2=v["r2"],
                          r2_uncentred=v.get("r2_uncentred", np.nan)) for k, v in FITS.items()])
print(fits.round(4).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
xx = np.linspace(1, 1100, 400)
for sp, c in [("human", "k"), ("pig", "tab:blue"), ("rat", "tab:red")]:
    t = PERI[sp]; f = FITS[sp, "i intercept"]
    ax[0].scatter(t.d_in, t.thk, s=8, c=c, label=sp); ax[0].plot(xx, f["intercept"] + f["slope"] * xx, c=c)
    ax[1].scatter(t.d_in, t.thk / t.d_in, s=8, c=c); ax[1].plot(xx, (f["intercept"] + f["slope"] * xx) / xx, c=c)
for a in ax:
    a.set_xlabel("fascicle diameter (um)")
ax[0].plot(xx, 0.03 * xx, "k--", label="3%"); ax[1].axhline(0.03, ls="--", c="k")
ax[0].set_ylabel("perineurium thickness (um)"); ax[1].set_ylabel("thickness / diameter")
ax[0].set_xlim(0, 1100); ax[0].set_ylim(0, 60); ax[1].set_xlim(0, 600); ax[1].set_ylim(0, 0.4); ax[0].legend()
allr = pd.concat([PERI[s] for s in ["human", "pig", "rat"]])
print("points outside the axes: left", int(((allr.d_in > 1100) | (allr.thk > 60)).sum()), "; right", int(((allr.d_in > 600) | (allr.thk / allr.d_in > 0.4)).sum()))
plt.show()

## human-perineurium-fit

> The fit of perineurium thickness against fascicle diameter for 73 human non-peanut fascicles has
> R2 = 0.75.

In [ ]:
for form in ["i intercept", "ii zero"]:
    f = FITS["human", form]
    print(f"{form}: n = {f['n']}, thickness = {f['slope']:.5f} d + {f['intercept']:.3f}, R2 = {f['r2']:.3f}" + (f", uncentred R2 = {f['r2_uncentred']:.3f}" if 'r2_uncentred' in f else ""))

**Result: reproduced.** 73 non-peanut human fascicles; the line with an intercept, the code's
`poly1`, gives R2 = 0.749 (thickness = 0.0370 d + 10.5 um). The zero-intercept line gives R2 = 0.17
about the mean (0.87 uncentred).

## pig-perineurium-fit

> The fit of perineurium thickness against fascicle diameter for 284 pig non-peanut fascicles (IF)
> has R2 = 0.30.

In [ ]:
for form in ["i intercept", "ii zero"]:
    f = FITS["pig", form]
    print(f"{form}: n = {f['n']}, thickness = {f['slope']:.5f} d + {f['intercept']:.3f}, R2 = {f['r2']:.3f}" + (f", uncentred R2 = {f['r2_uncentred']:.3f}" if 'r2_uncentred' in f else ""))
print("per level:", PERI["pig"].groupby("level").size().to_dict())

**Result: reproduced.** 284 non-peanut pig IF fascicles (with the hole trace of P13-3 counted as the
code counts it); the line with an intercept gives R2 = 0.298 (thickness = 0.0255 d + 3.44 um).

## rat-perineurium-fit

> The fit of perineurium thickness against fascicle diameter for 22 rat fascicles has R2 = 0.77.

In [ ]:
for sp in ["rat", "rat v1"]:
    t = PERI[sp]
    largest = t.loc[t.groupby("sample").a_out.idxmax()]
    for fs, tt in [("a all fascicles", t), ("b largest per sample", largest)]:
        for form, zero in [("i intercept", False), ("ii zero", True)]:
            f = pelot.line_fit(tt.d_in, tt.thk, zero=zero)
            print(f"{sp:6s} {fs:22s} {form:11s}: n = {f['n']:2d}, slope {f['slope']:.5f}, intercept {f['intercept']:.3f}, R2 = {f['r2']:.3f}")

**Result: reproduced.** With all 22 rat fascicles and a line with an intercept, the code's choice,
R2 = 0.765 (thickness = 0.0129 d + 1.37 um). The version 1 traces give R2 = 0.570 for the same
22 fascicles, so the paper's fit was made on the current traces.

## perineurium-species-order

> For a given fascicle diameter, human fascicles have the thickest perineurium and rat fascicles the
> thinnest.

In [ ]:
lo = max(PERI[s].d_in.min() for s in ["human", "pig", "rat"]); hi = min(PERI[s].d_in.max() for s in ["human", "pig", "rat"])
grid = np.linspace(lo, hi, 200)
pred = {s: FITS[s, "i intercept"]["intercept"] + FITS[s, "i intercept"]["slope"] * grid for s in ["human", "pig", "rat"]}
print(f"common range {lo:.1f} to {hi:.1f} um")
print("(a) human > pig > rat over the whole range:", bool(np.all(pred["human"] > pred["pig"]) and np.all(pred["pig"] > pred["rat"])))
for d in [lo, 250, hi]:
    print(f"   at {d:.0f} um: " + ", ".join(f"{s} {FITS[s, 'i intercept']['intercept'] + FITS[s, 'i intercept']['slope'] * d:.2f}" for s in ["human", "pig", "rat"]))
predz = {s: FITS[s, "ii zero"]["slope"] * grid for s in ["human", "pig", "rat"]}
print("zero-intercept fits, human > pig > rat over the range:", bool(np.all(predz["human"] > predz["pig"]) and np.all(predz["pig"] > predz["rat"])))
f1 = FITS["rat v1", "i intercept"]
print("variant, rat version 1 fit at 250 um:", round(f1["intercept"] + f1["slope"] * 250, 2))

**Result: reproduced.** Over the diameters common to the three species (32 to 284 um), the fitted
thickness is ordered human > pig > rat everywhere (reading a); at 250 um it is 19.8, 9.8 and 4.6 um
(reading b). The zero-intercept fits give the same order.

## perineurium-ratio-decreases

> The ratio of perineurium thickness to fascicle diameter decreases as fascicle diameter increases,
> particularly in humans and pigs.

In [ ]:
for s in ["human", "pig", "rat"]:
    t = PERI[s]; f = FITS[s, "i intercept"]
    d0, d1 = t.d_in.min(), t.d_in.max()
    r0, r1 = f["slope"] + f["intercept"] / d0, f["slope"] + f["intercept"] / d1
    rho, p = stats.spearmanr(t.d_in, t.thk / t.d_in)
    print(f"{s:6s} (a) fitted ratio {r0:.3f} at {d0:.0f} um to {r1:.3f} at {d1:.0f} um, decrease {r0 - r1:.3f}; (b) Spearman rho = {rho:.2f}, p = {p:.1e}")

**Result: reproduced.** Under reading (a) the ratio implied by each fit falls over the species'
range of diameters, by 1.54 in humans, 0.23 in pigs and 0.038 in rats, the human and pig decreases
larger than the rat's. Under reading (b) all three Spearman correlations are negative (-0.92, -0.60,
-0.78), the pig's weaker than the rat's.

## perineurium-thicker-larger-fascicles

> In the human and pig vagus nerves, larger fascicles have thicker perineurium.

In [ ]:
for s in ["human", "pig", "rat"]:
    t = PERI[s]; f = FITS[s, "i intercept"]
    rho, p = stats.spearmanr(t.d_in, t.thk)
    print(f"{s:6s} (a) slope {f['slope']:.4f} um/um (95% CI {f['slope_ci'][0]:.4f} to {f['slope_ci'][1]:.4f}); (b) Spearman rho = {rho:.2f}, p = {p:.1e}")

**Result: reproduced.** The fitted thickness rises with diameter in humans and pigs (slopes 0.037
and 0.026, confidence intervals above zero), and thickness correlates positively with diameter
(Spearman 0.92 and 0.54). The rat shows the same (slope 0.013, Spearman 0.86).

## perineurium-no-level-difference

> The relationship between perineurium thickness and fascicle diameter does not differ between
> cervical and subdiaphragmatic nerves in any of the three species.

In [ ]:
import statsmodels.formula.api as smf
for s in ["human", "pig", "rat", "rat v1"]:
    t = PERI[s]
    fc = pelot.line_fit(t[t.level == "C"].d_in, t[t.level == "C"].thk)
    fs = pelot.line_fit(t[t.level == "S"].d_in, t[t.level == "S"].thk)
    ov = lambda a, b: a[0] <= b[1] and b[0] <= a[1]
    a_ok = ov(fc["slope_ci"], fs["slope_ci"]) and ov(fc["intercept_ci"], fs["intercept_ci"])
    full = smf.ols("thk ~ d_in * level", data=t).fit(); red = smf.ols("thk ~ d_in", data=t).fit()
    Fv, pv, _ = full.compare_f_test(red)
    print(f"{s:6s} n C/S = {(t.level == 'C').sum()}/{(t.level == 'S').sum()}; cervical slope {fc['slope']:.4f} [{fc['slope_ci'][0]:.4f}, {fc['slope_ci'][1]:.4f}], "
          f"intercept {fc['intercept']:.2f} [{fc['intercept_ci'][0]:.2f}, {fc['intercept_ci'][1]:.2f}]; subdiaphragmatic slope {fs['slope']:.4f} "
          f"[{fs['slope_ci'][0]:.4f}, {fs['slope_ci'][1]:.4f}], intercept {fs['intercept']:.2f} [{fs['intercept_ci'][0]:.2f}, {fs['intercept_ci'][1]:.2f}]")
    print(f"        (a) bounds overlap for both coefficients: {a_ok}; (b) F test of level terms F = {Fv:.2f}, p = {pv:.4f}")
# Variant: the fascicles of one sample are not independent (human 73 fascicles from 18 samples,
# pig 284 from 8). Two tests of the level terms that allow for that: a mixed model with a random
# intercept per sample (REML, Wald test), and the pooled least squares fit with standard errors
# clustered by sample. Neither is the criterion's; they show how much the outcome leans on
# treating fascicles as independent.
import warnings as _w
for s in ["human", "pig", "rat"]:
    t = PERI[s].copy()
    hyp = "level[T.S] = 0, d_in:level[T.S] = 0"
    with _w.catch_warnings():
        _w.simplefilter("ignore")
        try:
            mm = smf.mixedlm("thk ~ d_in * level", t, groups=t["sample"]).fit(reml=True)
            wt = mm.wald_test(hyp, scalar=True)
            mm_txt = f"mixed model chi2 = {float(wt.statistic):.2f}, p = {float(wt.pvalue):.4f} (sample variance {float(mm.cov_re.iloc[0, 0]):.2f}, residual {mm.scale:.2f})"
        except Exception as e:
            mm_txt = f"mixed model failed: {type(e).__name__}"
        cl = smf.ols("thk ~ d_in * level", data=t).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(t["sample"])[0]})
        ct = cl.wald_test(hyp, scalar=True, use_f=True)
    print(f"{s:6s} samples {t['sample'].nunique()}: {mm_txt}; clustered least squares F = {float(ct.statistic):.2f}, p = {float(ct.pvalue):.4f}")

**Result: not reproduced (resting on the test chosen).** The cervical and subdiaphragmatic fits
differ in humans and in pigs: the 95% bounds of the slopes do not overlap (human 0.0248 to 0.0388
against 0.0494 to 0.0622; pig intercepts 4.14 to 6.39 against 2.67 to 3.65), and the F test of the
level terms gives F = 19.2 and F = 21.0 (p < 0.0001). In rats the two levels agree (bounds overlap,
F = 0.14, p = 0.87). These tests, the criterion's, treat fascicles as independent. Allowing for the
samples (variant), the human difference remains (mixed model with a random intercept per sample,
chi2 = 28.7, p < 0.0001; least squares with errors clustered by sample, F = 10.6, p = 0.0013), while
the pig difference does not (p = 0.58 and 0.26). So the outcome does not rest on independence alone:
the human fits differ under every test run. A reader is asked whether the criterion's tests read the
paper's "did not suggest".

## sunderland-reanalysis

> Refitting all 777 fascicles of Sunderland and Bradley (1952) gives a zero-intercept slope of 2.75%
> of fascicle diameter for perineurium thickness, with R2 = 0.51.

The 777 values are in sparc-65 (`files/code/Sunderland1984.mat`, "Data from Tables 1 to 4 in
Sunderland, S., Bradley, K.C., 1952", per `Sunderland1984_Plotting.m`).

In [ ]:
sund = pelot.read_mat("sparc-65", "files/code/Sunderland1984.mat")
x, y = sund["dfasc_um"].astype(float), sund["thk_peri_um"].astype(float)
f = pelot.line_fit(x, y, zero=True)
print(f"n = {len(x)}; d {x.min():.0f} to {x.max():.0f} um; thickness {y.min()} to {y.max()} um")
print(f"zero-intercept slope = {100 * f['slope']:.3f}%; R2 about the mean = {f['r2']:.3f}; uncentred R2 = {f['r2_uncentred']:.3f}")

**Result: reproduced.** On the 777 fascicles, the zero-intercept slope is 2.750% and R2 = 0.510
about the mean (reading a; the uncentred R2 is 0.853).

## rat-perineurium-near-3-percent

> Rat perineurium thickness is close to the published 3% of fascicle diameter, slightly thicker for
> smaller fascicles and thinner for larger ones.

In [ ]:
for sp in ["rat", "rat v1"]:
    t = PERI[sp]; f = FITS[sp, "i intercept"]
    g = np.linspace(t.d_in.min(), t.d_in.max(), 200)
    for ref in [0.03, 0.0275]:
        q = (f["intercept"] + f["slope"] * g) / (ref * g)
        print(f"{sp:6s} against {100 * ref:.2f}%: fit / line from {q[0]:.2f} at {g[0]:.0f} um to {q[-1]:.2f} at {g[-1]:.0f} um; "
              f"within 0.5 to 2: {bool(((q >= 0.5) & (q <= 2)).all())}; above at smallest {q[0] > 1}, below at largest {q[-1] < 1}")

**Result: reproduced.** The rat fit is 1.86 times the 3% line at the smallest rat fascicle (32 um)
and 0.59 times it at the largest (284 um), inside the factor of two and crossing the line once.
Against the 2.75% of the Sunderland reanalysis it runs from 2.02 to 0.64, just outside the factor
of two at the small end; the claim names 3%. The version 1 traces give 2.53 to 0.85.

## human-pig-perineurium-thicker

> Human and pig vagal perineurium is substantially thicker than the 3% of fascicle diameter found
> for human somatic nerves, especially for smaller fascicles.

In [ ]:
for s in ["human", "pig"]:
    t = PERI[s].copy(); t["ratio"] = t.thk / t.d_in
    med = t.d_in.median()
    small, large = t[t.d_in < med], t[t.d_in >= med]
    print(f"{s}: median ratio {t.ratio.median():.4f}; median excess over 0.03, below median diameter {(small.ratio - 0.03).median():.4f}, "
          f"above {(large.ratio - 0.03).median():.4f}; fascicles above 3%: {(t.ratio > 0.03).mean():.2f}")

**Result: reproduced.** Median thickness over diameter is 0.095 in humans and 0.053 in pigs, above
0.03, and the median excess over 0.03 is larger below the median diameter than above it (0.106
against 0.038 in humans, 0.033 against 0.016 in pigs).

## pig-if-mt-consistent

> Pig fascicle diameters from anti-fibronectin IF (14 to 303 um, mean 123, median 123, interquartile
> range 91 to 155 um) are consistent with those from MT (5 to 399 um, mean 135, median 130,
> interquartile range 96 to 168 um).

Quartiles with the Hazen definition, the one MATLAB's `prctile` uses.

In [ ]:
ifd = PERI["pig"].d_in.to_numpy()
mtd = pelot.fascicles(res, tab, "pig").d_in.to_numpy()
for name, d, exp in [("IF (non-peanut)", ifd, (14, 303, 123, 123, 91, 155)), ("MT (all traces)", mtd, (5, 399, 135, 130, 96, 168))]:
    q1, q3 = np.percentile(d, [25, 75], method="hazen")
    got = (d.min(), d.max(), d.mean(), np.median(d), q1, q3)
    print(f"{name}: n = {len(d)}; range {got[0]:.1f} to {got[1]:.1f}; mean {got[2]:.1f}; median {got[3]:.1f}; IQR {q1:.1f} to {q3:.1f}")
    print("   paper:", exp, "; mean and median within 10%:", abs(got[2] - exp[2]) <= 0.1 * exp[2] and abs(got[3] - exp[3]) <= 0.1 * exp[3],
          "; range and IQR within the widened bounds:", exp[0] * 0.9 <= got[0] and got[1] <= exp[1] * 1.1 and exp[4] * 0.9 <= q1 and q3 <= exp[5] * 1.1)
    print("   strict reading, each bound of the range and IQR within 10% of the paper's:",
          {n: abs(g - e) <= 0.1 * e for n, g, e in zip(["min", "max", "q1", "q3"], [got[0], got[1], q1, q3], [exp[0], exp[1], exp[4], exp[5]])})
print(f"MT traces of the version 1 masks of sparc-64 (data check): smallest {d_v1.min():.1f} um, mean {d_v1.mean():.1f}, median {np.median(d_v1):.1f}")

**Result: reproduced.** IF: 284 fascicles, 14.4 to 302.8 um, mean 123.3, median 123.1, IQR 88.0 to
152.6 um. MT: 766 traces, 23.7 to 399.1 um, mean 134.6, median 129.5, IQR 95.6 to 167.7 um. All
values are within the claim's tolerances, the ranges and IQRs inside the reported ones widened by
10%, as the criterion reads. Read strictly, each bound within 10% of the paper's, the MT minimum
fails (23.7 against 5 um): the version 4 masks hold no trace below 23.7 um, while the version 1
masks of sparc-64 have a smallest trace of 4.7 um (mean 137.6, median 131.8 um), so the paper's
5 um minimum probably comes from the version 1 masks.

## peanut-fascicle-counts

> Human IHC micrographs have 60 cervical inner perineurium traces (27 bundles in 10 peanut
> fascicles) and 44 subdiaphragmatic (4 in 2); pig IF micrographs have 210 cervical (57 in 25) and
> 179 subdiaphragmatic (48 in 19).

In [ ]:
rows = []
for sp, label in [("human", "human IHC"), ("pig_if", "pig IF")]:
    fa = pelot.fascicles(res, tab, sp)
    for lev in ["C", "S"]:
        t = fa[fa.level == lev]
        pe = t[t.peanut]
        rows.append(dict(data=label, level=LEVEL[lev], traces=len(t), peanut_bundles=len(pe),
                         peanut_fascicles=pe.groupby(["sample", "outer"]).ngroups, non_peanut=int((~t.peanut).sum())))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: human 60 (27 in 10), 44 (4 in 2); pig 210 (57 in 25), 179 (48 in 19)")
# per sample (the verifier asked for the pig IF cervical samples): peanut fascicles and bundles,
# outer traces with holes (bwboundaries would return them as extra outer traces), inner traces whose
# centroid lies in more than one outer trace (nested outers), and the non-peanut counts of
# PigIFMorphology.mat (pigIF_dfasc_notpeanut) for comparison
rows = []
for _, r in pelot.if_samples().iterrows():
    x = res["IF" + r["sample"]]; t = x["fasc"]
    pe = t[t.peanut]
    rows.append(dict(sample=r["sample"], level=r.level, traces=len(t), peanut_fascicles=pe.outer.nunique(), peanut_bundles=len(pe),
                     non_peanut=int((~t.peanut).sum()), outer_traces=x["n_outer"], outer_holes=x["holes_outer"], empty_outers=x["empty_outers"]))
pp = pd.DataFrame(rows)
fnp = []
for v in pm["pigIF_dfasc_notpeanut"]:  # one entry per pig (subjects 9, 11 to 15), an array per sample where a pig has two
    v = np.asarray(v)
    fnp += [len(np.atleast_1d(w)) for w in v] if v.dtype == object else [len(v)]
print(pp.to_string(index=False))
print("non-peanut counts per sample in PigIFMorphology.mat (file order):", fnp, "; total", sum(fnp), "; these masks", int(pp.non_peanut.sum()))

**Result: partially reproduced.** The four totals match (60, 44, 210 and 179), and so do the human
peanut counts (27 in 10, 4 in 2) and the pig subdiaphragmatic ones (48 in 19). The pig cervical
peanut bundles are 57, as in the paper, but in 24 peanut fascicles, not 25 (P9-3 6, P11-1 4, P12-1
8, P14-2 6). No outer trace has a hole, no inner centroid lies in two outer traces and none is
unmatched, so the matching is not ambiguous, and the non-peanut counts per sample (46, 36, 43, 29,
33, 28, 39, 30; total 284) are those of PigIFMorphology.mat. The deposited files do not record
which outer trace each peanut bundle belongs to, so the paper's 25 could not be traced.

## no-age-trend

> The morphology metrics show no trend with age in any species.

Ages are those of the code's sample lists (human years with 90+ entered as 99, pig weeks, rat days).
Per species and level, Pearson (reading a) and Spearman (reading b) correlations of each
per-sample metric with age.

In [ ]:
A = MAIN.copy()
A["mean_fasc_d"] = [pelot.fascicles(res, MAIN[MAIN["sample"] == s], sp).d_in.mean() if sp != "rat" else fd
                    for s, sp, fd in zip(A["sample"], A.species, A.fasc_d_largest)]
A["endo_pct"] = np.where(A.species == "rat", 100 * A.endo_area_largest / A.nerve_area, 100 * A.endo_area / A.nerve_area)
A["peri_pct"] = np.where(A.species == "pig", np.nan, np.where(A.species == "rat", 100 - A.endo_pct, 100 * A.peri_area / A.nerve_area))
rows = []
for (sp, lev), t in A.groupby(["species", "level"]):
    for mcol in ["nerve_d", "n_fasc", "mean_fasc_d", "endo_pct", "peri_pct"]:
        x, y = t.age_code.astype(float), t[mcol]
        if y.isna().all() or y.nunique() < 2:
            continue
        r, p = stats.pearsonr(x, y); rs, ps = stats.spearmanr(x, y)
        rows.append(dict(species=sp, level=lev, metric=mcol, n=len(t), ages=t.age_code.nunique(), pearson_r=r, pearson_p=p, spearman_rho=rs, spearman_p=ps))
AG = pd.DataFrame(rows)
print(AG.round(3).to_string(index=False))
print(f"\n{len(AG)} tests; Pearson p < 0.05: {(AG.pearson_p < 0.05).sum()}; Spearman p < 0.05: {(AG.spearman_p < 0.05).sum()}; "
      f"expected by chance at 0.05: {0.05 * len(AG):.1f}")
print(AG[(AG.pearson_p < 0.05) | (AG.spearman_p < 0.05)][["species", "level", "metric", "pearson_r", "pearson_p", "spearman_p"]].round(4).to_string(index=False))
from statsmodels.stats.multitest import multipletests
for col in ["pearson_p", "spearman_p"]:
    holm = multipletests(AG[col], method="holm")[1]; bh = multipletests(AG[col], method="fdr_bh")[1]
    print(f"variant, {col[:-2]} p adjusted over the {len(AG)} tests: smallest Holm {holm.min():.3f}, smallest Benjamini-Hochberg {bh.min():.3f}; "
          f"below 0.05: Holm {(holm < 0.05).sum()}, BH {(bh < 0.05).sum()}")
for sp, t in AG.groupby("species"):
    h = multipletests(t.pearson_p, method="holm")[1]
    print(f"   per species ({sp}, {len(t)} tests), Pearson Holm-adjusted smallest p {h.min():.3f}")
print("distinct ages per species:", MAIN.groupby("species").age_code.nunique().to_dict())

**Result: not reproduced (resting on the test chosen).** Of 27 Pearson correlations of a
per-sample metric with age, 3 have p < 0.05, all in pigs (cervical endoneurial proportion r = -0.72,
p = 0.029; subdiaphragmatic nerve diameter r = 0.75, p = 0.019; subdiaphragmatic mean fascicle
diameter r = 0.69, p = 0.039), against 1.4 expected by chance; with Spearman correlations 1 of 27
(human subdiaphragmatic mean fascicle diameter, p = 0.041). The criterion counts uncorrected
p < 0.05. Corrected over the 27 tests, none is below 0.05 (smallest Holm-adjusted p 0.52,
Benjamini-Hochberg 0.35; within pigs alone, Holm 0.15). The pig ages take four values (10.5 to 15
weeks). The paper's Supplement 6 plots the metrics against age (DataPlotting_v4.m draws them) and
reports no test. A reader is asked whether the criterion should correct for multiple tests.

## Micrographs judged by eye

The next three claims are judged by eye on the MT micrographs, as their criteria say. The images
are the JPEG 2000 derivatives of the nd2 micrographs, shown below; the readings are the author's,
recorded in `pelot.py` (`RAT_SHEATH_BUNDLES`, `HUMAN_SUB_FAT`, `PIG_FAT`, `PIG_CELL_BODIES`), and
printed with the images they were read from.

In [ ]:
def montage(ds, items, cols, step, title, size=3.2):
    rows = int(np.ceil(len(items) / cols))
    fig, axs = plt.subplots(rows, cols, figsize=(cols * size, rows * size * 0.75))
    for ax in np.ravel(axs):
        ax.axis("off")
    for ax, (label, path) in zip(np.ravel(axs), items):
        ax.imshow(pelot.thumbnail(ds, path, step)); ax.set_title(label, fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.show()

import io
from IPython.display import Image, display
def native(arr, box, step, title, marks=(), unsure=(), trace=None):
    """Show a region at `step` (1 = every pixel of the micrograph, 2 = every other) with one
    screen pixel per array pixel, as JPEG; marks are numbered (row, col), unsure open circles,
    trace a boolean mask of the region drawn in yellow."""
    r0, r1, c0, c1 = box
    if trace is not None:
        arr = arr.copy(); arr[trace[::step, ::step][:arr.shape[0], :arr.shape[1]]] = [255, 210, 0]
    h, w = arr.shape[:2]
    fig = plt.figure(figsize=(w / 100, h / 100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1]); ax.imshow(arr, extent=(c0, c1, r1, r0)); ax.axis("off")
    for n, (r, c) in enumerate(marks, start=1):
        ax.annotate(str(n), (c, r), color="yellow", fontsize=16, fontweight="bold", ha="center", va="center")
    for r, c in unsure:
        ax.plot(c, r, "o", mfc="none", mec="cyan", ms=22, mew=2)
    ax.text(0.01, 0.99, f"{title} (rows {r0} to {r1}, columns {c0} to {c1}, every {step} px)", transform=ax.transAxes,
            va="top", fontsize=11, color="k", bbox=dict(fc="w", alpha=0.7))
    buf = io.BytesIO(); fig.savefig(buf, format="jpeg", dpi=100, pil_kwargs=dict(quality=80)); plt.close(fig)
    display(Image(buf.getvalue()))

## rat-carotid-sheath-bundles

> The rat cervical carotid sheath holds up to six fascicles, of which the largest, taken as the
> vagus, was segmented.

In [ ]:
smp16 = pelot.read_excel("sparc-16", "files/samples.xlsx")
loc16 = {(a, b): str(l) for a, b, l in zip(smp16.subject_id, smp16.sample_id, smp16["specimen anatomical location"])}
cerv = [(f"R{a[4:]}-{b[4:]}", p) for a, b, p in pelot.micrograph_paths("sparc-16") if "cervical" in loc16.get((a, b), "")]
left = [x for x in cerv if "left" in loc16[("sub-" + x[0][1:].split("-")[0], "sam-" + x[0].split("-")[1])]]
right = [x for x in cerv if x not in left]
montage("sparc-16", right, 3, 6, "right cervical (thumbnails, every 6th pixel)")
# the nine analysed left sections at half resolution of the derivative, with the bundles
# counted (numbered; 1 is the traced vagus) and the profiles not counted (open circles)
for s, p in left:
    rd = pelot.RAT_SHEATH_FULLRES[s]
    native(pelot.crop("sparc-16", p, rd["box"], 2), rd["box"], 2, f"{s}: {len(rd['bundles'])} bundles", rd["bundles"], rd["unsure"])
b = pd.Series(pelot.RAT_SHEATH_BUNDLES)
an = [s for s in b.index if s in set(rat[rat.level == "C"]["sample"])]
print("bundles per sample (by eye; left at full resolution, right on thumbnails):", b.to_dict())
print("left counts equal the marked lists:", all(b[s] == len(pelot.RAT_SHEATH_FULLRES[s]["bundles"]) for s in an))
print("profiles not counted (could not be told from a vessel):", {s: len(v["unsure"]) for s, v in pelot.RAT_SHEATH_FULLRES.items()})
print(f"(a) analysed left cervical: max {b[an].max()}, samples with >1 bundle {int((b[an] > 1).sum())} of {len(an)}")
print(f"(b) all 18 cervical: max {b.max()}, samples with >1 bundle {int((b > 1).sum())} of {len(b)}")
print("traced vagus is the largest bundle in every analysed sample (by eye):", all(pelot.RAT_VAGUS_LARGEST.values()))

**Result: reproduced (by eye, resting on the counting rule).** At full resolution (shown at half
resolution of the derivatives, bundles numbered), the nine analysed left cervical sheaths hold 2,
5, 5, 2, 3, 5, 4, 6 and 2 bundles (R16-3 to R25-3); the largest count is six (R24-3), every section
has more than one bundle, and the bundle traced as the vagus (1) is the largest in each. Eight
small profiles that could not be told from a vessel at this resolution (open circles) are not
counted; counting them would raise R17-3 and R18-1 to six and no section above six (R24-3 has
none). The outcome rests on these calls: had two small profiles of R24-3 (3 and 6) been vessels,
the largest count would be five unless R17-3's open circle is a bundle. The first reading, on thumbnails, found at most
five; the extra bundles are small profiles (for example R24-3 at row 3400, column 1250). The right
sections keep their thumbnail counts (at most five), so reading (b) also gives six.
Supplementary Figure 16, which shows the paper's count, was not available.

## human-subdiaphragmatic-fatty-tissue

> Pig nerves have consistent collagenous epineurium, whereas many human subdiaphragmatic nerves have
> fatty extrafascicular tissue.

In [ ]:
smp61 = pelot.read_excel("sparc-61", "files/samples.xlsx")
loc61 = {(a, b): str(l).lower() for a, b, l in zip(smp61.subject_id, smp61.sample_id, smp61["specimen anatomical location"])}
hs = [(f"C{a[4:]}-{b[4:]}", p) for a, b, p in pelot.micrograph_paths("sparc-61") if "subdiaphragmatic" in loc61.get((a, b), "")]
montage("sparc-61", hs, 6, 8, "human subdiaphragmatic (sparc-61)", size=2.6)
pg = [(f"P{a[4:]}-{b[4:]}", p) for a, b, p in pelot.micrograph_paths("sparc-82")]
montage("sparc-82", pg, 6, 8, "pig, all levels (sparc-82)", size=2.6)
# full-resolution regions read as adipose tissue in analysed pig samples, with the nerve trace of
# sparc-64 (yellow) drawn over them: fat inside the trace is in the epineurium as the paper segmented it
from skimage.segmentation import find_boundaries
mp82 = {(f"P{a[4:]}-{b[4:]}"): p for a, b, p in pelot.micrograph_paths("sparc-82")}
crops = pelot.PIG_CROPS["fat"]
inside_frac = {}
for s, box in crops.items():
    sub, sam = s[1:].split("-")
    nm = pelot.read_mask("sparc-64", f"files/primary/sub-{sub}/sam-{sam}/sub-{sub}_sam-{sam}_{s}NerveMask.tif")
    r0, r1, c0, c1 = box
    e = ndi.binary_dilation(find_boundaries(nm[r0:r1, c0:c1], mode="inner"), iterations=3)
    step = 2 if (r1 - r0) * (c1 - c0) > 1.5e6 else 1
    native(pelot.crop("sparc-82", mp82[s], box, step), box, step, f"{s}, nerve trace of sparc-64 in yellow", trace=e)
    inside_frac[s] = round(float(nm[r0:r1, c0:c1].mean()), 2)
print("fraction of each region inside the nerve trace:", inside_frac)
hf = pd.Series(pelot.HUMAN_SUB_FAT)
print(f"human subdiaphragmatic micrographs read: {len(hf)} of {len(hs)} shown; with adipose tissue: {int(hf.sum())} ({hf.mean():.2f})")
print(f"pig micrographs: {len(pg)}; with adipose tissue within the epineurium: {len(pelot.PIG_FAT)}:",
      sorted(f"P{a[4:]}-{b[4:]}" for a, b in pelot.PIG_FAT))
print("readings of 'many': (a) at least half", hf.mean() >= 0.5, "; (b) at least a third", hf.mean() >= 1 / 3,
      "; pig rule, no more than one of 45:", len(pelot.PIG_FAT) <= 1)

**Result: not reproduced (by eye, resting on the criterion's rule).** Adipose tissue is read in 29
of the 32 human subdiaphragmatic micrographs, which meets both readings of "many", but also within
the epineurium of 13 of the 45 pig micrographs, where the criterion allows at most one. The
full-resolution views show unilocular adipocytes inside the authors' own nerve traces of sparc-64 in
P11-1, P13-1, P5-1, P9-3 and P8-7 (65 to 84% of each region shown lies inside the trace; in P5-1
the fat lies between fascicles). Pig epineurium is more collagenous than the human subdiaphragmatic
tissue, as the paper says, but not free of fat.

## pig-cervical-cell-bodies

> In two pig cervical cross sections (P11-1 and P13-1) one fascicle contains cell bodies.

In [ ]:
mp82 = {(f"P{a[4:]}-{b[4:]}"): p for a, b, p in pelot.micrograph_paths("sparc-82")}
pc = [(s, mp82[s]) for s in pig[pig.level == "C"]["sample"]]
montage("sparc-82", pc, 3, 2, "analysed pig cervical sections, half resolution", size=6)
# full resolution: the fascicles read as holding cell bodies (P11-1, P13-1) and the pale
# fascicles of P14-2 that the half-resolution view could not call
for s, box in pelot.PIG_CROPS["cell_bodies"].items():
    native(pelot.crop("sparc-82", mp82[s], box), box, 1, s)
cb = pd.Series(pelot.PIG_CELL_BODIES)
print("fascicle with cell bodies (by eye):", cb.to_dict())
print("P11-1 and P13-1:", bool(cb["P11-1"] and cb["P13-1"]), "; other analysed cervical samples with one:", int(cb.drop(["P11-1", "P13-1"]).sum()))

**Result: reproduced (by eye, resting on the reading).** At full resolution, P11-1 (the fascicle at
the upper edge) and P13-1 (the large fascicle at the left end) each have one fascicle holding large
round somata with nuclei, unlike the fibre texture of the others. The pale fascicles of P14-2 that
the half-resolution view could not call hold fibre profiles and no somata, and no other analysed
cervical section shows a fascicle with cell bodies.

## Literature values of Figure 7

Supplement 4, the spreadsheet of literature values, was not supplied with the text read. The
paper's plotting code, `files/code/DataPlotting_v4.m` (deposited in sparc-60, 64 and 65),
hard-codes them ("Stats in lit (see Supplementary Excel spreadsheet)") as the means, SDs and
reference numbers that Figure 7 plots, in the order of the Figure 7 legend. They are parsed from
the file in sparc-65 (`pelot.literature_values`); the studies are matched to the code's reference
numbers by that order. The paper's summary statistics are recomputed from the traces as in Figure 7
(rat: largest fascicle).

In [ ]:
LIT = pelot.literature_values()
print(LIT.to_string(index=False))
print("\nFigure 7 legend order checked: human cervical diameters", LIT.query("species == 'human' and level == 'C' and stat == 'd_nerve_mm'").ref.to_list(),
      "; human cervical fascicle counts", LIT.query("species == 'human' and level == 'C' and stat == 'n_fasc'").ref.to_list())
OURS = MAIN.assign(A_nerve_mm2=MAIN.nerve_area / 1e6, d_nerve_mm=MAIN.nerve_d / 1000,
                   A_endo_mm2=np.where(MAIN.species == "rat", MAIN.endo_area_largest, MAIN.endo_area) / 1e6,
                   A_endo_pct=100 * np.where(MAIN.species == "rat", MAIN.endo_area_largest, MAIN.endo_area) / MAIN.nerve_area)
def ours(sp, lev, stat):
    x = OURS[(OURS.species == sp) & (OURS.level == lev)][stat]
    return x.mean(), x.std(), x.min(), x.max()
def compare(sp, lev, stat):
    m, sd, lo, hi = ours(sp, lev, stat)
    L = LIT[(LIT.species == sp) & (LIT.level == lev) & (LIT.stat == stat)]
    t = L[["study", "mean", "sd"]].copy()
    t["within_our_mean_sd"] = (t["mean"] >= m - sd) & (t["mean"] <= m + sd)
    t["ours_below"] = m < t["mean"]
    t["ranges_overlap"] = (lo <= t["mean"] + t["sd"]) & (hi >= t["mean"] - t["sd"])
    print(f"{sp} {LEVEL[lev]} {stat}: ours {m:.4g} +/- {sd:.3g} (range {lo:.4g} to {hi:.4g}), n = 9")
    print(t.round(4).to_string(index=False))
    return t

## literature-comparison-human-pig

> Human cervical fascicle numbers and endoneurial areas match prior reports while the nerve areas
> and diameters are generally smaller, so the endoneurial proportion is larger; human
> subdiaphragmatic sizes match prior reports, with fewer fascicles than one study of the posterior
> branch; pig fascicle numbers match literature, with slightly smaller mean diameters.

The criterion's rules: "comparable" each literature mean within our mean +/- SD; "smaller" our mean
below the literature mean; "overlapping ranges" our sample range overlapping the literature mean
+/- SD; "fewer fascicles" the human subdiaphragmatic mean below Stakenborg et al. (2020); "larger
proportion" the human cervical endoneurial percentage above the two literature percentages.

In [ ]:
parts = {}
t = compare("human", "C", "n_fasc"); parts["human cervical fascicle number comparable"] = t.within_our_mean_sd.all()
t = compare("human", "C", "A_endo_mm2"); parts["human cervical endoneurial area comparable"] = t.within_our_mean_sd.all()
t1 = compare("human", "C", "A_nerve_mm2"); t2 = compare("human", "C", "d_nerve_mm")
parts["human cervical nerve area and diameter smaller (all studies)"] = t1.ours_below.all() and t2.ours_below.all()
print(f"   smaller than {int(t1.ours_below.sum())} of {len(t1)} areas and {int(t2.ours_below.sum())} of {len(t2)} diameters")
parts["human cervical ranges overlap"] = t1.ranges_overlap.all() and t2.ranges_overlap.all()
parts["human cervical diameter below 3.0 mm (Discussion)"] = ours("human", "C", "d_nerve_mm")[0] < 3.0
t = compare("human", "C", "A_endo_pct"); parts["human cervical endoneurial proportion larger"] = t.ours_below.eq(False).all()
t1 = compare("human", "S", "A_nerve_mm2"); t2 = compare("human", "S", "d_nerve_mm")
parts["human subdiaphragmatic area and diameter comparable (all studies)"] = t1.within_our_mean_sd.all() and t2.within_our_mean_sd.all()
ant = [0, 2]  # anterior VN rows of Tailai et al. and Gravgaard (legend order: anterior, posterior)
print("   anterior VN studies only: areas within", t1.iloc[ant].within_our_mean_sd.to_list(), "; diameters within", t2.iloc[ant].within_our_mean_sd.to_list())
t = compare("human", "S", "n_fasc"); parts["human subdiaphragmatic fewer fascicles than Stakenborg (posterior)"] = (~t.ours_below).sum() == 0 and t.ours_below.all()
for lev in "CS":
    t = compare("pig", lev, "n_fasc"); parts[f"pig {LEVEL[lev]} fascicle number comparable"] = t.within_our_mean_sd.all()
    t = compare("pig", lev, "d_nerve_mm"); parts[f"pig {LEVEL[lev]} mean diameter smaller"] = t.ours_below.all()
print()
for k, v in parts.items():
    print(f"{'holds' if v else 'FAILS'}: {k}")
print(f"\n{sum(parts.values())} of {len(parts)} parts hold")

**Result: partially reproduced (resting on the criterion's rules).** Of the 12 parts, 9 hold: the
human cervical fascicle numbers (all 7 literature means within 6.7 +/- 4.5), smaller human cervical
nerve areas and diameters (our 3.05 mm2 and 1.92 mm below all 7 studies' means), the diameter below
3.0 mm, the larger endoneurial proportion (32.2% against 10.8 and 12.9%), fewer human
subdiaphragmatic fascicles than Stakenborg et al. (4.9 against 16), pig fascicle numbers (46 and 43
within 46.9 +/- 10.3 and 38.2 +/- 7.1) and smaller pig diameters (2.20 and 1.64 against 2.93 and
2.09 mm). Three fail: the human cervical endoneurial area (0.92 +/- 0.34 mm2; the two Seki et al.
means, 1.32 and 1.84, lie outside), the overlapping ranges (the six literature diameters given
without an SD lie above our largest, 2.55 mm, though 6 of 7 areas overlap), and the human
subdiaphragmatic sizes, where the posterior-branch values of Tailai et al. (3.80 mm2, 2.20 mm) and
Gravgaard (5.03 mm2, 2.53 mm) lie outside our mean +/- SD; the anterior-branch values, which the
paper singles out, lie inside.

## rat-literature-comparison

> Rat cervical nerve diameters and areas agree with published data, while the rat subdiaphragmatic
> nerves are slightly larger than prior data.

The criterion: each published cervical mean within the rat cervical mean +/- SD, and the rat
subdiaphragmatic mean above each published subdiaphragmatic mean. Rat nerve diameter and area are
those of the outer perineurium of the largest fascicle (rat-nerve-diameter), as in Figure 7.

In [ ]:
rp = {}
for stat in ["d_nerve_mm", "A_nerve_mm2"]:
    t = compare("rat", "C", stat); rp[f"cervical {stat}: published means within ours +/- SD"] = (int(t.within_our_mean_sd.sum()), len(t))
    t = compare("rat", "S", stat); rp[f"subdiaphragmatic {stat}: ours above every published mean"] = (int((~t.ours_below).sum()), len(t))
for k, (a, n) in rp.items():
    print(f"{k}: {a} of {n}")
# variant: the version 1 traces of sparc-60
for lev in "CS":
    x = tab_v1[tab_v1.level == lev].nerve_d / 1000
    L = LIT.query("species == 'rat' and level == @lev and stat == 'd_nerve_mm'")["mean"]
    print(f"variant, version 1 traces, {LEVEL[lev]} diameter {x.mean():.4f} +/- {x.std():.4f} mm; published within: {int(((L >= x.mean() - x.std()) & (L <= x.mean() + x.std())).sum())} of {len(L)}; ours above: {int((x.mean() > L).sum())} of {len(L)}")

**Result: partially reproduced (resting on the criterion's rules).** The rat subdiaphragmatic mean
(0.135 mm, 0.0145 mm2) is above all four values of Prechtl and Powley (1985), so that part holds.
The cervical part does not: of the three published means, only Licursi de Alcantara et al.'s left
VN (0.281 mm, 0.0620 mm2) lies within our 0.258 +/- 0.025 mm and 0.053 +/- 0.010 mm2; Fazan and
Lachat (0.226 mm) is below and the right VN of Licursi de Alcantara et al. (0.299 mm, 0.0703 mm2)
above. The version 1 traces give the same counts.

In [ ]:
print(f"notebook ran in {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("files read (dataset, path, version, sha256):")
for (ds, path, ver), h in sorted(pelot.READ_LOG.items()):
    print(f"  {ds} v{ver} {path} {h}")

## Summary

| Claim | Outcome |
|---|---|
| rat-sample-set, human-sample-set, pig-mt-sample-set, pig-if-sample-set, mt-micrograph-collections | reproduced |
| nerve-diameter-anova | reproduced (unbounded variance, JMP's default) |
| human-pig-nerve-diameter-similar, rat-nerve-diameter, rat-ten-times-smaller, nerve-cervical-larger | reproduced |
| fascicle-number-human-pig, rat-subdiaphragmatic-multifascicular, rat-multifascicular-proportions | reproduced |
| fascicle-diameter-anova, rat-fascicle-diameter, cervical-fascicle-diameter-species, subdiaphragmatic-fascicle-diameter-comparable, fascicles-larger-cervical | reproduced |
| rat-diameter-like-fascicle, human-pig-tissue-proportions, pig-if-endoneurium-lower, rat-level-ratios, human-pig-level-ratios, human-variability, pig-oblong-human-round | reproduced |
| human-subdiaphragmatic-fatty-tissue | not reproduced (by eye; pig nerve traces also hold fat) |
| pig-cervical-cell-bodies | reproduced (by eye) |
| rat-carotid-sheath-bundles | reproduced (by eye at full resolution; at most six bundles, R24-3) |
| literature-comparison-human-pig | partially reproduced (9 of 12 parts; literature values from DataPlotting_v4.m) |
| rat-literature-comparison | partially reproduced (subdiaphragmatic larger; 1 of 3 cervical values within the SD) |
| perineurium-species-order, perineurium-ratio-decreases, human-perineurium-fit, pig-perineurium-fit, rat-perineurium-fit, perineurium-thicker-larger-fascicles | reproduced |
| perineurium-no-level-difference | not reproduced (human and pig fits differ between levels on the criterion's tests; human also with sample clustering) |
| sunderland-reanalysis | reproduced |
| rat-perineurium-near-3-percent, human-pig-perineurium-thicker, pig-if-mt-consistent | reproduced |
| peanut-fascicle-counts | partially reproduced (pig cervical: 57 bundles in 24 peanut fascicles, not 25) |
| no-age-trend | not reproduced (3 of 27 uncorrected Pearson tests below 0.05, all pig; none after correction) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The mixed models are refitted in Python by REML with
Kenward-Roger degrees of freedom, JMP's method, implemented here; F values come within 0.2 to 9% of
the paper's and p values agree in significance, though the per-species fascicle contrasts of pigs
and rats are weaker than the paper's (fascicles-larger-cervical). Morphometry follows the paper's
deposited MATLAB code, emulated (boundary areas by Pick's theorem, with object traces 8-connected
and hole traces 4-connected as `bwboundaries` makes them, masks stored as MinIsWhite inverted as
MATLAB reads them); the human values match the deposited output file exactly. The rat fascicle the
criteria name, the largest, differs in two samples from the one the paper's code keeps, the first
in `bwboundaries` order, which reproduces the paper's rat values; both are shown. Three claims are
judged by eye on the JPEG 2000 derivatives, now at full resolution where the readings matter, and
two no-difference claims rest on tests the paper does not name. The literature values come from
the paper's plotting code, as Supplement 4 was not available.

What was found about the datasets. The rat traces of sparc-60 changed between version 1 and
version 2 (outer perineurium boundaries moved inward); the paper's perineurium results match the
current traces, while the output metrics file deposited in sparc-60 and sparc-65 was computed from
version 1, and gives the connective tissue boundary as the nerve diameter of the three
multifascicular rat samples. The pig fascicle diameters in that file are about 1.4 times those
measured from the pig masks and reported in the paper, by a factor that varies between fascicles,
and its pig fascicle counts are those of the version 1 pig masks in all 18 samples. Several binary
masks are stored with photometric MinIsWhite, so that readers that take the raw bits get inverted
masks. sparc-60's readme gives the rats' ages as 65 to 268 days, the subjects sheet 75 to 268. In
sparc-61 the micrograph of C48-2 is filed under sub-48/sam-2, which has no row in samples.xlsx,
while the sheet lists sub-48/sam-1 with no file; copies of C51-3 and C63-1 are also filed under
other subjects' folders. About the paper: its code keeps the first rat fascicle where the Methods
say the largest.